# 🖋️ Arabic Handwriting OCR — v5 النهائية المُحسَّنة

**المميزات الجديدة المدمجة:**
* قاموس تصحيح ذكي: ثقته، سياقاته، مرات استخدامه.
* واجهة مراجعة للقاموس (تبويب جديد) مع مؤشرات بصرية 🔴🟡🟢⏳.
* عتبات ديناميكية قابلة للضبط + معايرة إحصائية تلقائية.
* تراجع عن التصحيحات مع تحديث القاموس.


In [ ]:
# ── تثبيت على Colab ──
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q \
    pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm

# ── Manjaro/Arch (محلي) — أزل علامة # ──
# sudo pacman -S python-pip poppler tesseract tesseract-data-ara
# pip install --user pdf2image easyocr pyspellchecker langdetect transformers peft \
#   huggingface_hub datasets Pillow opencv-python-headless pandas numpy matplotlib \
#   scikit-learn pytesseract ar-corrector gradio openpyxl jiwer torch torchvision \
#   tensorboard albumentations tqdm

In [ ]:
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
import socket, fcntl
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import gradio as gr
import easyocr
from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from tqdm import tqdm

# مدقق عربي (اختياري)
try:
    from ar_corrector.corrector import Corrector
    _AR_OK = True
except ImportError:
    _AR_OK = False

# Colab setup
try:
    from google.colab import drive, userdata
    # Check if Drive is already mounted. If so, unmount it first to ensure a clean state.
    if os.path.ismount('/content/drive'):
        print("ℹ️ Google Drive is already mounted. Attempting to unmount for a clean re-mount.")
        drive.flush_and_unmount() # This explicitly unmounts

    # Explicitly clear the mount directory if it's not a mount point and contains files.
    # This handles stubborn cases where flush_and_unmount might not fully clear the directory.
    if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
        print("⚠️ /content/drive still contains files after unmount attempt. Clearing directory contents.")
        for item in os.listdir('/content/drive'):
            item_path = os.path.join('/content/drive', item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
            except OSError as e:
                print(f"Error clearing {item_path}: {e}")

    # Now attempt to mount. force_remount=True will handle cases where the mount point
    # might contain stale files or isn't properly unmounted.
    drive.mount('/content/drive', force_remount=True)

    try:
        _HF_TOKEN = userdata.get('HF_TOKEN')
        if _HF_TOKEN:
            os.environ['HF_TOKEN'] = _HF_TOKEN
        print('✅ Colab: Drive + HF_TOKEN')
    except Exception:
        _HF_TOKEN = None
    IN_COLAB = True
except ImportError:
    _HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ بيئة محلية')

DetectorFactory.seed = 0
print('✅ الاستيرادات اكتملت')

In [ ]:
@dataclass
class Config:
    """
    الإعدادات المركزية — مدمجة من جميع الإصدارات + تحسينات الذاكرة.

    وضع الذاكرة memory_mode:
        'low'  — للأجهزة ≤16GB RAM (batch=1, FP32, lazy loading)
        'high' — لـ Colab بـ GPU / أجهزة 32GB+
    """

    # ====== المسارات ======
    project_root: str  = '/content/drive/MyDrive/Handwritten_OCR_Ultimate'
    pdf_path: str      = '/content/drive/MyDrive/input.pdf'
    db_name: str       = 'handwriting_data.db'

    # ====== النماذج ======
    trocr_model_name: str = 'David-Magdy/TR_OCR_LARGE'
    hf_token: str         = ''
    hf_dataset_repo: str  = ''   # مثال: DrAbdulmalek/handwriting-ocr
    ocr_languages: list   = field(default_factory=lambda: ['en', 'ar'])

    # ====== وضع الذاكرة — الأهم ======
    memory_mode: str   = 'auto'    # 'low' | 'high' | 'auto'
    use_gpu: bool      = True
    use_fp16: bool     = True      # نصف الدقة على GPU → توفير 50% ذاكرة

    # ====== أداء OCR ======
    # batch_size يُضبط تلقائياً حسب memory_mode:
    #   low  → 1  |  high → 8  |  Colab GPU → 16
    trocr_batch_size: int      = 4
    num_beams: int             = 4
    max_text_length: int       = 64
    easy_conf_threshold: float = 0.80   # تخطي TrOCR إذا EasyOCR واثق
    trocr_default_conf: float  = 0.70
    low_conf_threshold: float  = 0.50

    # ====== Preprocessing ======
    dpi: int            = 300
    clahe_clip: float   = 2.0
    clahe_tile: tuple   = (8, 8)
    denoise_h: int      = 20
    enable_deskew: bool = True
    min_word_w: int     = 15
    min_word_h: int     = 10
    dilation_kernel: tuple = (25, 5)

    # ====== التصحيح ======
    correction_min_votes: int = 1

    # ====== الصفحات ======
    pages_start: int = 1
    pages_end: int   = 5

    # ====== Fine-tuning ======
    finetune_min_samples: int  = 50
    finetune_epochs: int       = 5
    finetune_batch_size: int   = 4
    finetune_lr: float         = 1e-5
    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.1
    lora_targets: list         = field(default_factory=lambda: ['query', 'value'])

    # ====== Active Learning ======
    al_min_corrections: int    = 50
    al_reprocess_limit: int    = 100

    # ====== المزامنة ======
    sync_lock_timeout: int     = 30

    # ====== التصدير ======
    export_val_ratio: float    = 0.1
    auto_export: bool          = True

    # ====== Gradio ======
    gradio_share: bool = True
    gradio_port: int   = 7860

    # ====== Logging ======
    log_level: str = 'INFO'

    # ====== إعدادات العتبات الجديدة للمؤشر البصري ======
    dict_thresholds: dict = field(default_factory=lambda: {
        "conf_low": 0.60,
        "conf_mid": 0.80,
        "usage_high": 50,
        "usage_mid": 20,
        "days_critical": 30,
        "days_warning": 14,
        "new_days_warning": 3,
        "calibrate_method": "percentile"
    })

    # ==================== Properties ====================
    @property
    def root(self):         return Path(self.project_root)
    @property
    def db_path(self):      return self.root / 'database' / self.db_name
    @property
    def logs_dir(self):     return self.root / 'logs'
    @property
    def exports_dir(self):  return self.root / 'exports'
    @property
    def cache_dir(self):    return self.root / 'models_cache'
    @property
    def artifacts_dir(self):return self.root / 'artifacts'
    @property
    def backups_dir(self):  return self.root / 'backups'
    @property
    def tb_dir(self):       return self.root / 'runs'
    @property
    def feedback_csv(self): return self.logs_dir / 'user_corrections_feedback.csv'
    @property
    def stats_json(self):   return self.logs_dir / 'processing_stats.json'
    @property
    def correction_dict_path(self): return self.artifacts_dir / 'correction_dict.json'
    @property
    def checkpoint_file(self): return self.artifacts_dir / 'ocr_checkpoint.json'
    @property
    def metrics_log(self):  return self.logs_dir / 'metrics_history.csv'
    @property
    def runs_csv(self):     return self.logs_dir / 'runs_history.csv'
    @property
    def lora_path(self):    return self.cache_dir / 'trocr_lora_finetuned'
    @property
    def lock_file(self):    return self.artifacts_dir / 'ocr.lock'
    @property
    def sync_status(self):  return self.artifacts_dir / 'sync_status.json'
    @property
    def study_guide_dir(self): return self.exports_dir / 'study_guide'

    def resolve_memory_mode(self):
        """تحديد وضع الذاكرة تلقائياً"""
        if self.memory_mode != 'auto':
            return self.memory_mode
        if torch.cuda.is_available():
            vram = torch.cuda.get_device_properties(0).total_memory // 1e9
            return 'high' if vram >= 8 else 'low'
        # CPU only
        import psutil
        ram_gb = psutil.virtual_memory().total / 1e9
        return 'high' if ram_gb >= 24 else 'low'

    def setup(self):
        """إنشاء جميع المجلدات وتهيئة الملفات الأساسية"""
        for d in [self.root/'database', self.logs_dir, self.exports_dir,
                  self.cache_dir, self.artifacts_dir, self.backups_dir,
                  self.tb_dir, self.study_guide_dir]:
            d.mkdir(parents=True, exist_ok=True)
        if self.hf_token:
            os.environ['HF_TOKEN'] = self.hf_token
        os.environ['TRANSFORMERS_CACHE'] = str(self.cache_dir)
        os.environ['TORCH_HOME']         = str(self.cache_dir)
        for csv_path, cols in [
            (self.feedback_csv, ['timestamp','image_id','original_text','corrected_text','status']),
            (self.runs_csv, ['run_id','timestamp','pages','words','avg_conf','duration_sec','status']),
        ]:
            if not csv_path.exists():
                pd.DataFrame(columns=cols).to_csv(csv_path, index=False, encoding='utf-8-sig')

        # ضبط batch_size تلقائياً
        mode = self.resolve_memory_mode()
        if self.trocr_batch_size == 4:   # لم يُغيَّر يدوياً
            self.trocr_batch_size = 16 if mode == 'high' else 1
        print(f'💾 وضع الذاكرة: {mode} | batch_size={self.trocr_batch_size}')

    def setup_easyocr_symlink(self):
        """ربط نماذج EasyOCR بـ Drive لتوفير إعادة التحميل"""
        local = Path.home() / '.EasyOCR'
        drive_p = self.root / '.EasyOCR'
        if not drive_p.exists():
            drive_p.mkdir(parents=True, exist_ok=True)
        if local.exists() and not local.is_symlink():
            shutil.move(str(local), str(drive_p))
        if not local.exists():
            try:
                os.symlink(drive_p, local)
            except Exception:
                pass

    @classmethod
    def for_colab(cls, pdf_name='input.pdf', hf_token='', hf_repo='',
                  memory_mode='auto'):
        return cls(
            project_root='/content/drive/MyDrive/Handwritten_OCR_Ultimate',
            pdf_path=f'/content/drive/MyDrive/{pdf_name}',
            hf_token=hf_token or _HF_TOKEN or '',
            hf_dataset_repo=hf_repo,
            memory_mode=memory_mode,
        )

    @classmethod
    def for_local(cls, base_dir='~/handwriting_ocr', pdf_path='./input.pdf',
                  memory_mode='low'):
        """إعدادات البيئة المحلية مع تحسينات الذاكرة"""
        return cls(
            project_root=str(Path(base_dir).expanduser()),
            pdf_path=str(Path(pdf_path).expanduser()),
            memory_mode=memory_mode,
            use_fp16=False,       # CPU لا يدعم FP16 عادةً
            dpi=200,              # دقة أقل = ذاكرة أقل
            pages_end=3,          # صفحات أقل في الاختبار
        )


# ====== إنشاء CFG ======
# للـ Colab:
CFG = Config.for_colab(pdf_name='python notes.pdf')
# للمحلي — احذف التعليق:
# CFG = Config.for_local(base_dir='~/handwriting_ocr', pdf_path='./input.pdf')

CFG.setup()
CFG.setup_easyocr_symlink()
print('✅ Config جاهز (Enhanced):', CFG.root)

In [ ]:
_LOG_FILE = CFG.logs_dir / f'ocr_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
_EVENTS   = CFG.logs_dir / 'ocr_events.jsonl'

LOGGER = logging.getLogger('HandwrittenOCR')
LOGGER.setLevel(getattr(logging, CFG.log_level))
LOGGER.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for _h in [
    RotatingFileHandler(_LOG_FILE, maxBytes=2_000_000, backupCount=5, encoding='utf-8'),
    logging.StreamHandler()
]:
    _h.setFormatter(_fmt)
    LOGGER.addHandler(_h)


def log_event(event_type: str, payload: dict = None, level: str = 'info'):
    payload = payload or {}
    entry = {'ts': datetime.now().isoformat(), 'event': event_type, 'data': payload}
    with open(_EVENTS, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    getattr(LOGGER, level, LOGGER.info)(
        f'{event_type} | {json.dumps(payload, ensure_ascii=False)}')

print('✅ اللوغات جاهزة')

In [ ]:
class FileLock:
    """
    قفل ملفات لمنع التعارضات عند العمل المتزامن من جهازين.
    يستخدم fcntl على Linux/macOS و msvcrt على Windows.
    """

    def __init__(self, lock_path, timeout=30):
        self.lock_path = Path(lock_path)
        self.timeout   = timeout
        self._file     = None

    def acquire(self):
        self.lock_path.parent.mkdir(parents=True, exist_ok=True)
        self._file = open(self.lock_path, 'w')
        start = time.time()
        while True:
            try:
                if platform.system() == 'Windows':
                    import msvcrt
                    msvcrt.locking(self._file.fileno(), msvcrt.LK_NBLCK, 1)
                else:
                    fcntl.flock(self._file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
                info = {'pid': os.getpid(), 'host': socket.gethostname(),
                        'ts': datetime.now().isoformat()}
                self._file.seek(0); self._file.truncate()
                self._file.write(json.dumps(info)); self._file.flush()
                LOGGER.debug(f'FileLock acquired: {self.lock_path}')
                return True
            except (BlockingIOError, OSError):
                if time.time() - start > self.timeout:
                    self._file.close(); self._file = None
                    raise TimeoutError(
                        'تعذر الحصول على قفل الملف — جهاز آخر يعمل حالياً. '
                        'حاول بعد قليل أو أوقف المعالجة من الجهاز الآخر.')
                time.sleep(0.5)

    def release(self):
        if self._file is None:
            return
        try:
            if platform.system() == 'Windows':
                import msvcrt; msvcrt.locking(self._file.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(self._file.fileno(), fcntl.LOCK_UN)
        except Exception:
            pass
        finally:
            try: self._file.close()
            except Exception: pass
            self._file = None

    def __enter__(self): self.acquire(); return self
    def __exit__(self, *_): self.release(); return False

print('✅ FileLock جاهز')

In [ ]:
class HandwritingDB:
    """
    مدير SQLite — مخطط v3:
    - يدعم run_id, raw_text, created_at, updated_at
    - ترقية تلقائية من مخطط v1/v2
    - دعم multi-device عبر FileLock
    """

    SCHEMA = '''
        CREATE TABLE IF NOT EXISTS handwriting_data (
            image_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            image_data     BLOB    NOT NULL,
            predicted_text TEXT    DEFAULT '',
            raw_text       TEXT    DEFAULT '',
            status         TEXT    DEFAULT 'unverified',
            confidence     REAL    DEFAULT 0.0,
            model_source   TEXT    DEFAULT 'none',
            x INTEGER DEFAULT 0, y INTEGER DEFAULT 0,
            w INTEGER DEFAULT 0, h INTEGER DEFAULT 0,
            page_num       INTEGER DEFAULT 0,
            run_id         TEXT    DEFAULT '',
            created_at     TEXT    DEFAULT '',
            updated_at     TEXT    DEFAULT ''
        );
        CREATE TABLE IF NOT EXISTS processing_runs (
            run_id           TEXT PRIMARY KEY,
            started_at       TEXT,
            ended_at         TEXT,
            input_path       TEXT,
            pages_processed  INTEGER DEFAULT 0,
            words_processed  INTEGER DEFAULT 0,
            avg_confidence   REAL    DEFAULT 0.0,
            status           TEXT    DEFAULT 'running'
        );
        CREATE TABLE IF NOT EXISTS review_events (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp      TEXT,
            image_id       INTEGER,
            original_text  TEXT,
            corrected_text TEXT,
            action         TEXT,
            reviewer       TEXT DEFAULT 'user'
        );
        CREATE INDEX IF NOT EXISTS idx_status ON handwriting_data(status);
        CREATE INDEX IF NOT EXISTS idx_page   ON handwriting_data(page_num);
        CREATE INDEX IF NOT EXISTS idx_conf   ON handwriting_data(confidence);
        CREATE INDEX IF NOT EXISTS idx_run    ON handwriting_data(run_id);
    '''

    def __init__(self, db_path: Path):
        self.db_path = db_path
        db_path.parent.mkdir(parents=True, exist_ok=True)
        with self._conn() as c:
            c.executescript(self.SCHEMA)
            self._migrate(c)
        LOGGER.info(f'DB جاهزة: {db_path}')

    def _conn(self):
        conn = sqlite3.connect(self.db_path, check_same_thread=False)
        conn.execute('PRAGMA journal_mode=WAL')   # أداء أفضل مع multi-writer
        conn.execute('PRAGMA synchronous=NORMAL') # توازن بين السرعة والأمان
        return conn

    def _migrate(self, conn):
        """ترقية مخطط v1/v2 → v3 بشكل آمن"""
        cur = conn.execute("PRAGMA table_info(handwriting_data)")
        existing = {r[1] for r in cur.fetchall()}
        new_cols = {
            'raw_text':   "TEXT DEFAULT ''",
            'run_id':     "TEXT DEFAULT ''",
            'created_at': "TEXT DEFAULT ''",
            'updated_at': "TEXT DEFAULT ''",
        }
        for col, typedef in new_cols.items():
            if col not in existing:
                conn.execute(f"ALTER TABLE handwriting_data ADD COLUMN {col} {typedef}")
        conn.execute("UPDATE handwriting_data SET status='verified'   WHERE status='yes'")
        conn.execute("UPDATE handwriting_data SET status='unverified' WHERE status='no'")
        conn.commit()

    # ---------- Insert ----------
    def insert_word(self, image_data, predicted_text, raw_text='',
                    status='unverified', confidence=0.0, model_source='none',
                    x=0, y=0, w=0, h=0, page_num=0, run_id='') -> int:
        ts = datetime.now().isoformat()
        with self._conn() as c:
            cur = c.execute(
                '''INSERT INTO handwriting_data
                   (image_data,predicted_text,raw_text,status,confidence,
                    model_source,x,y,w,h,page_num,run_id,created_at,updated_at)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                (image_data, predicted_text, raw_text, status, confidence,
                 model_source, x, y, w, h, page_num, run_id, ts, ts))
            c.commit()
            return cur.lastrowid

    # ---------- Update / Delete ----------
    def update_word(self, image_id, predicted_text=None, status=None):
        sets, vals = ['updated_at=?'], [datetime.now().isoformat()]
        if predicted_text is not None:
            sets.insert(0, 'predicted_text=?'); vals.insert(0, predicted_text)
        if status is not None:
            sets.insert(0, 'status=?'); vals.insert(0, status)
        vals.append(image_id)
        with self._conn() as c:
            c.execute(f"UPDATE handwriting_data SET {','.join(sets)} WHERE image_id=?", vals)
            c.commit()

    def delete_word(self, image_id: int):
        with self._conn() as c:
            c.execute('DELETE FROM handwriting_data WHERE image_id=?', (image_id,))
            c.commit()

    def delete_pages(self, start, end) -> int:
        with self._conn() as c:
            cur = c.execute(
                'DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?', (start, end))
            c.commit()
            return cur.rowcount

    # ---------- Queries ----------
    def _rows(self, sql, params=()):
        with self._conn() as c:
            c.row_factory = sqlite3.Row
            return [dict(r) for r in c.execute(sql, params).fetchall()]

    def get_all(self):
        return self._rows('SELECT * FROM handwriting_data ORDER BY page_num,y,x')

    def get_unverified(self):
        return self._rows(
            "SELECT * FROM handwriting_data WHERE status='unverified' "
            "ORDER BY confidence ASC")

    def get_verified(self):
        return self._rows(
            "SELECT * FROM handwriting_data "
            "WHERE status IN ('verified','sentence_corrected') ORDER BY image_id")

    def get_low_confidence(self, threshold=0.5, limit=100):
        return self._rows(
            "SELECT * FROM handwriting_data WHERE confidence<? "
            "ORDER BY confidence ASC LIMIT ?", (threshold, limit))

    def count_by_status(self) -> dict:
        rows = self._rows(
            'SELECT status, COUNT(*) as cnt FROM handwriting_data GROUP BY status')
        return {r['status']: r['cnt'] for r in rows}

    def insert_run(self, run_id, input_path):
        with self._conn() as c:
            c.execute(
                'INSERT OR REPLACE INTO processing_runs '
                '(run_id,started_at,input_path,status) VALUES (?,?,?,?)',
                (run_id, datetime.now().isoformat(), str(input_path), 'running'))
            c.commit()

    def finish_run(self, run_id, pages, words, avg_conf):
        with self._conn() as c:
            c.execute(
                'UPDATE processing_runs SET ended_at=?,pages_processed=?,'
                'words_processed=?,avg_confidence=?,status=? WHERE run_id=?',
                (datetime.now().isoformat(), pages, words, avg_conf, 'completed', run_id))
            c.commit()

    def log_review(self, image_id, original, corrected, action):
        with self._conn() as c:
            c.execute(
                'INSERT INTO review_events '
                '(timestamp,image_id,original_text,corrected_text,action) VALUES (?,?,?,?,?)',
                (datetime.now().isoformat(), image_id, original, corrected, action))
            c.commit()

    def get_last_review_event(self):
        """يحصل على آخر حدث مراجعة مؤهل للتراجع عنه"""
        with self._conn() as c:
            c.row_factory = sqlite3.Row
            # نركز على أحداث 'verified' التي تحتوي على image_id
            row = c.execute(
                "SELECT * FROM review_events WHERE action IN ('verified') AND image_id IS NOT NULL ORDER BY timestamp DESC LIMIT 1"
            ).fetchone()
            return dict(row) if row else None

    def delete_review_event(self, event_id: int):
        """يحذف حدث مراجعة بناءً على معرفه"""
        with self._conn() as c:
            c.execute('DELETE FROM review_events WHERE id=?', (event_id,))
            c.commit()


DB = HandwritingDB(CFG.db_path)
print('✅ قاعدة البيانات جاهزة')

In [ ]:
class DataMigrator:
    """
    ترحيل البيانات من النسخ القديمة (v1/v2/v3/Pro/Integrated) إلى v5.
    يرحّل: قواعد البيانات + ملفات التصحيحات + يُعيد بناء القاموس.
    """

    OLD_FOLDERS = [
        'Handwriting_Dataset', 'Handwritten_OCR_Integrated',
        'Handwritten_OCR_Pro', 'Handwritten_OCR',
    ]

    def __init__(self, config: Config):
        self.cfg = config

    def scan(self, base_path='') -> dict:
        """فحص سريع للمشاريع القديمة المتاحة"""
        base = Path(base_path or self.cfg.root.parent)
        found = []
        for name in self.OLD_FOLDERS:
            p = base / name
            if not p.exists():
                continue
            info = {'name': name, 'path': str(p),
                    'db_files': list(str(f) for f in p.rglob('*.db')),
                    'feedback_files': []}
            for pat in ['*feedback*.csv', '*correction*.csv']:
                info['feedback_files'].extend(str(f) for f in p.rglob(pat))
            if info['db_files'] or info['feedback_files']:
                found.append(info)
        return {'projects': found, 'total': len(found)}

    def migrate(self, base_path='', verified_only=True) -> dict:
        """ترحيل شامل مع سجل مفصل"""
        base = Path(base_path or self.cfg.root.parent)
        stats = {'db_records': 0, 'feedback': 0, 'dict_entries': 0, 'errors': []}

        print('🔍 جاري البحث عن بيانات قديمة...')
        for name in self.OLD_FOLDERS:
            folder = base / name
            if not folder.exists():
                continue
            for db_file in folder.rglob('*.db'):
                try:
                    n = self._migrate_db(str(db_file), name, verified_only)
                    stats['db_records'] += n
                    if n:
                        print(f'  📥 {name}: {n} سجل')
                except Exception as e:
                    stats['errors'].append(str(e))
                    LOGGER.warning(f'migrate_db {db_file}: {e}')

        stats['feedback'] = self._merge_feedback(base)
        stats['dict_entries'] = self._rebuild_dict()
        print(f'✅ الترحيل اكتمل: {stats}')
        return stats

    def _migrate_db(self, src_path: str, label: str, verified_only: bool) -> int:
        src = sqlite3.connect(src_path)
        tgt = sqlite3.connect(self.cfg.db_path, check_same_thread=False)
        migrated = 0
        try:
            cur = src.execute("PRAGMA table_info(handwriting_data)")
            src_cols = {r[1] for r in cur.fetchall()}
            where = "WHERE status IN ('verified','sentence_corrected','yes')" if verified_only else ""
            cols = [c for c in ['image_data','predicted_text','raw_text','status',
                                 'confidence','model_source','x','y','w','h','page_num']
                    if c in src_cols]
            rows = src.execute(f"SELECT {','.join(cols)} FROM handwriting_data {where}").fetchall()
            now = datetime.now().isoformat()
            tgt.execute('PRAGMA journal_mode=WAL')
            tgt_cur = tgt.cursor()
            for row in rows:
                try:
                    d = dict(zip(cols, row))
                    if d.get('status') == 'yes': d['status'] = 'verified'
                    elif d.get('status') == 'no': d['status'] = 'unverified'
                    # Skip if likely duplicate
                    dup = tgt_cur.execute(
                        "SELECT 1 FROM handwriting_data WHERE predicted_text=? AND page_num=? LIMIT 1",
                        (d.get('predicted_text',''), d.get('page_num', 0))).fetchone()
                    if dup:
                        continue
                    tgt_cur.execute(
                        '''INSERT INTO handwriting_data
                           (image_data,predicted_text,raw_text,status,confidence,
                            model_source,x,y,w,h,page_num,run_id,created_at,updated_at)
                           VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                        (d.get('image_data',b''),
                         d.get('predicted_text',''),
                         d.get('raw_text', d.get('predicted_text','')),
                         d.get('status','unverified'),
                         d.get('confidence', 0.0),
                         d.get('model_source','none'),
                         d.get('x',0), d.get('y',0),
                         d.get('w',0), d.get('h',0),
                         d.get('page_num',0),
                         f'migrated_{label}', now, now))
                    migrated += 1
                except Exception:
                    continue
            tgt.commit()
        finally:
            src.close(); tgt.close()
        return migrated

    def _merge_feedback(self, base: Path) -> int:
        all_dfs = []
        for name in self.OLD_FOLDERS:
            folder = base / name
            if not folder.exists(): continue
            for pat in ['*feedback*.csv', '*correction*.csv']:
                for f in folder.rglob(pat):
                    try:
                        df = pd.read_csv(f, encoding='utf-8-sig')
                        if {'original_text','corrected_text'}.issubset(df.columns):
                            all_dfs.append(df)
                    except Exception:
                        pass
        if not all_dfs:
            return 0
        merged = pd.concat(all_dfs, ignore_index=True)
        merged = merged.dropna(subset=['original_text','corrected_text'])
        merged = merged[merged['original_text'].astype(str).str.strip() != '']
        merged = merged[merged['original_text'] != merged['corrected_text']]
        merged = merged.drop_duplicates(subset=['original_text','corrected_text'], keep='last')
        # دمج مع الملف الحالي
        if self.cfg.feedback_csv.exists():
            existing = pd.read_csv(self.cfg.feedback_csv, encoding='utf-8-sig')
            merged = pd.concat([existing, merged], ignore_index=True)
            merged = merged.drop_duplicates(
                subset=['original_text','corrected_text'], keep='last')
        merged.to_csv(self.cfg.feedback_csv, index=False, encoding='utf-8-sig')
        return len(merged)

    def _rebuild_dict(self) -> int:
        if not self.cfg.feedback_csv.exists():
            return 0
        df = pd.read_csv(self.cfg.feedback_csv, encoding='utf-8-sig')
        if df.empty: return 0
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o = str(row.get('original_text','')).strip()
            c = str(row.get('corrected_text','')).strip()
            if o and c and o != c:
                buckets[o][c] += 1
        result = {o: cnt.most_common(1)[0][0]
                  for o, cnt in buckets.items()
                  if cnt.most_common(1)[0][1] >= self.cfg.correction_min_votes}
        self.cfg.correction_dict_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2), 'utf-8')
        return len(result)


MIGRATOR = DataMigrator(CFG)
print('✅ DataMigrator جاهز')
# لترحيل البيانات القديمة نفّذ:
# MIGRATOR.migrate()

In [ ]:
def deskew(gray: np.ndarray) -> np.ndarray:
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50: return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3: return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess_image(img_bgr: np.ndarray, adaptive=False):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim==3 else img_bgr.copy()
    if CFG.enable_deskew: gray = deskew(gray)
    gray = cv2.createCLAHE(clipLimit=CFG.clahe_clip, tileGridSize=CFG.clahe_tile).apply(gray)
    gray = cv2.fastNlMeansDenoising(gray, h=CFG.denoise_h)
    if adaptive:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary, gray


def _iou(b1, b2) -> float:
    x1,y1,w1,h1 = b1; x2,y2,w2,h2 = b2
    xi1,yi1 = max(x1,x2), max(y1,y2)
    xi2,yi2 = min(x1+w1,x2+w2), min(y1+h1,y2+h2)
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    union = w1*h1+w2*h2-inter
    return inter/union if union>0 else 0


def smart_segmentation(img_bgr, binary, detections=None) -> list:
    if detections:
        boxes = []
        for det in detections:
            pts = np.array(det[0], dtype=np.int32)
            x,y,w,h = cv2.boundingRect(pts)
            if w>CFG.min_word_w and h>CFG.min_word_h:
                boxes.append((x,y,w,h))
        if boxes: return boxes
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, CFG.dilation_kernel)
    dilated = cv2.dilate(binary, kernel, iterations=1)
    contours,_ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [(x,y,w,h) for c in contours
             for x,y,w,h in [cv2.boundingRect(c)]
             if w>CFG.min_word_w and h>CFG.min_word_h]
    return sorted(boxes, key=lambda b: (b[1],b[0]))


def match_boxes_detections(boxes, detections) -> list:
    if not detections: return [(b, None) for b in boxes]
    det_boxes = []
    for det in detections:
        pts = np.array(det[0], dtype=np.int32)
        det_boxes.append((cv2.boundingRect(pts), det))
    result, used = [], set()
    for box in boxes:
        best_det, best_iou = None, 0
        for i, (db, det) in enumerate(det_boxes):
            if i in used: continue
            iou = _iou(box, db)
            if iou > best_iou and iou > 0.3:
                best_iou, best_det = iou, det
                used.add(i)
        result.append((box, best_det))
    return result


def crop_safe(img, x, y, w, h):
    H,W = img.shape[:2]
    return img[max(0,y):min(H,y+h), max(0,x):min(W,x+w)]

print('✅ Preprocessing جاهز')

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() and CFG.use_gpu else 'cpu')
LOGGER.info(f'Device: {DEVICE}')

print(f'⏳ تحميل النماذج على {DEVICE}...')
_t0 = time.time()

# تحسين الذاكرة على CPU
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, os.cpu_count() or 2))
    LOGGER.info(f'CPU threads: {torch.get_num_threads()}')

_hf_kw = {'cache_dir': str(CFG.cache_dir)}
if CFG.hf_token:
    _hf_kw['token'] = CFG.hf_token

# Clear cache to avoid I/O errors during model loading from Google Drive
if CFG.cache_dir.exists():
    print(f'⚠️ مسح ذاكرة التخزين المؤقت للنماذج: {CFG.cache_dir}')
    shutil.rmtree(CFG.cache_dir)
    CFG.cache_dir.mkdir(parents=True, exist_ok=True) # Recreate empty directory

# Temporarily set TRANSFORMERS_CACHE to a local /tmp directory
# to avoid potential I/O issues when downloading to Google Drive
original_transformers_cache = os.environ.get('TRANSFORMERS_CACHE')
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_transformers_cache'
Path('/tmp/hf_transformers_cache').mkdir(parents=True, exist_ok=True)
print(f'ℹ️ تم تحويل TRANSFORMERS_CACHE مؤقتًا إلى: {os.environ["TRANSFORMERS_CACHE"]}')

try:
    trocr_processor = TrOCRProcessor.from_pretrained(CFG.trocr_model_name, **_hf_kw)
    trocr_model     = VisionEncoderDecoderModel.from_pretrained(CFG.trocr_model_name, **_hf_kw)
finally:
    # Restore original TRANSFORMERS_CACHE after model loading attempt
    if original_transformers_cache:
        os.environ['TRANSFORMERS_CACHE'] = original_transformers_cache
    else:
        del os.environ['TRANSFORMERS_CACHE']
    print(f'ℹ️ TRANSFORMERS_CACHE تم إرجاعه إلى وضعه الأصلي.')

# FP16 على GPU فقط (توفير ~50% ذاكرة)
if DEVICE.type == 'cuda' and CFG.use_fp16:
    trocr_model = trocr_model.half()
    print('✅ FP16 مفعّل → توفير 50% من ذاكرة GPU')
trocr_model = trocr_model.to(DEVICE)

# تحميل LoRA إذا وُجد
if CFG.lora_path.exists():
    try:
        from peft import PeftModel
        trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_path)).to(DEVICE)
        print('✅ LoRA محمّلة')
    except Exception as e:
        print(f'⚠️ LoRA: {e}')

# EasyOCR
try:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=(DEVICE.type=='cuda'))
except Exception:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=False)

_ar_corrector  = Corrector() if _AR_OK else None
_en_spellcheck = SpellChecker(language='en')
print(f'✅ النماذج جاهزة في {time.time()-_t0:.1f}s')


def normalize_text(x) -> str:
    return '' if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x).strip()


def detect_lang(text: str) -> str:
    try:
        return detect(str(text)) if text and text.strip() else 'unknown'
    except Exception:
        return 'unknown'


def spell_correct(text: str) -> str:
    text = normalize_text(text)
    if not text: return ''
    try:
        lang = detect_lang(text)
        if lang == 'ar' and _ar_corrector:
            return _ar_corrector.contextual_correct(text)
        return ' '.join(_en_spellcheck.correction(w) or w for w in text.split())
    except Exception:
        return text


def trocr_batch_predict(crops: list) -> list:
    """
    Batch inference مع:
    - Beam search لدقة أعلى
    - تنظيف الذاكرة بعد كل batch
    - FP16 على GPU
    """
    if not crops: return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops]
    try:
        enc = trocr_processor(images=pil_imgs, return_tensors='pt', padding=True)
        pv  = enc.pixel_values
        if CFG.use_fp16 and DEVICE.type == 'cuda':
            pv = pv.half()
        pv = pv.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(
                pv,
                max_length=CFG.max_text_length,
                num_beams=CFG.num_beams,
                early_stopping=True,
                length_penalty=1.0,
            )
        texts = trocr_processor.batch_decode(ids, skip_special_tokens=True)
        # تنظيف
        del pv, ids, enc
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        return texts
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            LOGGER.warning('OOM — تقليل batch size تلقائياً')
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
            # تراجع: معالجة واحدة واحدة
            results = []
            for crop in crops:
                try:
                    r = trocr_batch_predict([crop])
                    results.extend(r)
                except Exception:
                    results.append('')
            return results
        LOGGER.warning(f'trocr_batch: {e}')
        return [''] * len(crops)
    except Exception as e:
        LOGGER.warning(f'trocr_batch: {e}')
        return [''] * len(crops)

print('✅ OCR Engine جاهز')

In [ ]:
# -*- coding: utf-8 -*-
"""الخلية 10: قاموس التصحيح المُوسَّع (CorrectionRule) + دوال البناء والمعايرة"""

from dataclasses import dataclass

@dataclass
class CorrectionRule:
    """قاعدة تصحيح مع ميتاداتا للمراجعة والمساءلة"""
    original: str
    correction: str
    votes: int = 1
    first_seen: str = field(default_factory=lambda: datetime.now().isoformat())
    last_used: str = None
    usage_count: int = 0
    last_reviewed: str = None
    reviewer: str = None
    confidence: float = 1.0
    contexts: list = field(default_factory=list)
    flagged: bool = False
    notes: str = ""

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if v is not None}

    @classmethod
    def from_dict(cls, data, original: str):
        if isinstance(data, str):
            return cls(original=original, correction=data)
        return cls(original=original, **{k: v for k, v in data.items() if k != 'original'})


def build_correction_dict_v2() -> dict:
    """بناء قاموس التصحيح مع الميتاداتا من ملف التغذية الراجعة."""
    if not CFG.feedback_csv.exists():
        return {}
    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty:
            return {}
        buckets = defaultdict(list)
        for _, row in df.iterrows():
            o = normalize_text(row.get('original_text'))
            c = normalize_text(row.get('corrected_text'))
            if o and c and o != c:
                buckets[o].append({
                    'correction': c,
                    'timestamp': row.get('timestamp'),
                    'image_id': row.get('image_id'),
                    'page': row.get('page_num')
                })
        result = {}
        for original, corrections in buckets.items():
            counter = Counter(c['correction'] for c in corrections)
            best_correction, best_votes = counter.most_common(1)[0]
            total_votes = sum(counter.values())
            confidence = best_votes / total_votes if total_votes > 0 else 0
            contexts = []
            result[original] = CorrectionRule(
                original=original,
                correction=best_correction,
                votes=best_votes,
                usage_count=len(corrections),
                confidence=round(confidence, 3),
                contexts=contexts,
                flagged=confidence < 0.7 or best_votes < CFG.correction_min_votes
            )
        CFG.correction_dict_path.write_text(
            json.dumps({k: v.to_dict() for k, v in result.items()}, ensure_ascii=False, indent=2),
            'utf-8')
        return result
    except Exception as e:
        LOGGER.error(f'build_correction_dict_v2: {e}')
        return {}


def load_correction_dict_v2() -> dict:
    if CFG.correction_dict_path.exists():
        try:
            with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
                raw = json.load(f)
            return {k: CorrectionRule.from_dict(v, k) for k, v in raw.items()}
        except Exception:
            return build_correction_dict_v2()
    return build_correction_dict_v2()


def apply_corrections_v2(text: str, rules: dict) -> str:
    """تطبيق التصحيحات باستخدام القاموس المُوسَّع"""
    if not text:
        return ''
    return ' '.join(rules.get(tok, tok) if isinstance(rules.get(tok), str) 
                    else rules[tok].correction if tok in rules else tok
                    for tok in text.split())


def track_correction_usage(original: str, applied_correction: str):
    """تتبع استخدام قاعدة تصحيح وتحديث last_used و usage_count"""
    if not CFG.correction_dict_path.exists():
        return
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        if original not in raw:
            return
        rule = CorrectionRule.from_dict(raw[original], original)
        rule.usage_count += 1
        rule.last_used = datetime.now().isoformat()
        raw[original] = rule.to_dict()
        with open(CFG.correction_dict_path, 'w', encoding='utf-8') as f:
            json.dump(raw, f, ensure_ascii=False, indent=2)
    except Exception as e:
        LOGGER.debug(f'track_usage: {e}')


# ====== دوال المعايرة الإحصائية ======
def auto_calibrate_dict_thresholds(method=None):
    """حساب عتبات المؤشرات البصرية إحصائياً من بيانات القاموس"""
    if not CFG.correction_dict_path.exists():
        return {"error": "لا توجد قواعد في القاموس"}
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        rules = [CorrectionRule.from_dict(v, k) for k, v in data.items()]
        if len(rules) < 10:
            return {"error": f"البيانات غير كافية ({len(rules)} قاعدة). الحد الأدنى: 10"}
        method = method or CFG.dict_thresholds["calibrate_method"]
        confs = np.array([r.confidence for r in rules])
        usages = np.array([r.usage_count for r in rules])
        if method == "std_dev":
            c_low = max(0.0, float(np.mean(confs) - np.std(confs)))
            c_mid = float(np.mean(confs))
            u_high = float(np.mean(usages) + np.std(usages))
            u_mid = float(np.mean(usages))
        else:  # percentile
            c_low, c_mid = np.percentile(confs, [25, 50])
            u_mid, u_high = np.percentile(usages, [75, 90])
        CFG.dict_thresholds.update({
            "conf_low": round(c_low, 2),
            "conf_mid": round(c_mid, 2),
            "usage_high": round(u_high, 0),
            "usage_mid": round(u_mid, 0)
        })
        return {
            "status": "calibrated", "method": method, "samples": len(rules),
            "thresholds": {k: CFG.dict_thresholds[k] for k in ["conf_low", "conf_mid", "usage_high", "usage_mid"]}
        }
    except Exception as e:
        return {"error": str(e)}


def calculate_rule_indicator(rule: CorrectionRule) -> dict:
    """حساب المؤشر البصري لقاعدة تصحيح (🔴🟡🟢⏳)"""
    th = CFG.dict_thresholds
    now = datetime.now()
    days_review = (now - datetime.fromisoformat(rule.last_reviewed)).days if rule.last_reviewed else 999
    days_added = (now - datetime.fromisoformat(rule.first_seen)).days
    score = 0
    if rule.confidence < th["conf_low"]:
        score += 3
    elif rule.confidence < th["conf_mid"]:
        score += 2
    if rule.flagged:
        score += 2
    if rule.usage_count > th["usage_high"] and days_review > th["days_critical"]:
        score += 2
    elif rule.usage_count > th["usage_mid"] and days_review > th["days_warning"]:
        score += 1
    if days_review > th["days_critical"] or (days_review == 999 and days_added > th["new_days_warning"]):
        score += 1
    if score >= 5:
        visual, color = "🔴 عاجل", "#d32f2f"
    elif score >= 3:
        visual, color = "🟡 مراجعة مقترحة", "#f57c00"
    elif score == 0 and days_review == 999:
        visual, color = "⏳ جديد", "#607d8b"
    else:
        visual, color = "🟢 موثوق", "#2e7d32"
    summary = (f"📊 الثقة: {rule.confidence:.0%} | 🔁 الاستخدام: {rule.usage_count}× | "
               f"🕒 آخر مراجعة: {'منذ ' + str(days_review) + ' يوم' if days_review < 999 else 'لم تُراجع'}")
    return {"indicator": visual, "color": color, "summary": summary, "score": score}


# ====== التوافق مع الكود القديم ======
# نحتفظ بالدوال القديمة للتوافق مع process_document الأصلي
def build_correction_dict() -> dict:
    """توافق: يبني القاموس البسيط"""
    if not CFG.feedback_csv.exists():
        return {}
    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty:
            return {}
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o = normalize_text(row.get('original_text'))
            c = normalize_text(row.get('corrected_text'))
            if o and c and o != c:
                buckets[o][c] += 1
        result = {o: cnt.most_common(1)[0][0]
                  for o, cnt in buckets.items()
                  if cnt.most_common(1)[0][1] >= CFG.correction_min_votes}
        CFG.correction_dict_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2), 'utf-8')
        return result
    except Exception as e:
        LOGGER.error(f'correction_dict: {e}')
        return {}


def load_correction_dict() -> dict:
    """توافق: يحمل القاموس البسيط"""
    if CFG.correction_dict_path.exists():
        try:
            return json.loads(CFG.correction_dict_path.read_text('utf-8'))
        except Exception:
            return build_correction_dict()
    return build_correction_dict()


def apply_corrections(text: str, d: dict) -> str:
    return ' '.join(d.get(tok, tok) for tok in text.split()) if text else ''


def append_feedback(image_id, original, corrected, status='verified'):
    row = [{
        'timestamp': datetime.now().isoformat(),
        'image_id': image_id,
        'original_text': original,
        'corrected_text': corrected,
        'status': status,
    }]
    pd.DataFrame(row).to_csv(CFG.feedback_csv, mode='a', header=False,
                             index=False, encoding='utf-8-sig')


# ====== تهيئة القاموس ======
correction_dict_rules = load_correction_dict_v2()
correction_dict = {k: v.correction if hasattr(v, 'correction') else v
                   for k, v in correction_dict_rules.items()}
print(f'✅ قاموس التصحيح المُوسَّع: {len(correction_dict_rules)} إدخال | بسيط: {len(correction_dict)} إدخال')

In [ ]:
def save_checkpoint(data: dict):
    CFG.checkpoint_file.write_text(
        json.dumps(data, ensure_ascii=False, indent=2), 'utf-8')

def load_checkpoint():
    return (json.loads(CFG.checkpoint_file.read_text('utf-8'))
            if CFG.checkpoint_file.exists() else None)

def clear_checkpoint():
    if CFG.checkpoint_file.exists(): CFG.checkpoint_file.unlink()

print('✅ Checkpoint جاهز')

In [ ]:
def load_pages(input_path, start_page=None, end_page=None):
    input_path = str(input_path)
    ext = Path(input_path).suffix.lower()
    sp  = start_page or CFG.pages_start
    ep  = end_page   or CFG.pages_end
    if ext == '.pdf':
        imgs = convert_from_path(input_path, dpi=CFG.dpi, first_page=sp, last_page=ep)
        pages = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in imgs]
        del imgs
        return pages, list(range(sp, sp + len(pages)))
    if ext in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}:
        img = cv2.imread(input_path)
        if img is None: raise ValueError(f'تعذر قراءة: {input_path}')
        return [img], [1]
    raise ValueError(f'نوع غير مدعوم: {ext}')


def process_document(input_path=None, start_page=None, end_page=None,
                     resume=True, adaptive=False, progress_cb=None) -> dict:
    """
    المعالجة الرئيسية مع:
    - قفل الملفات (multi-device safe)
    - Checkpoint استئناف
    - Batch TrOCR مع conditional inference
    - تنظيف الذاكرة بعد كل صفحة
    - تصدير تلقائي
    """
    global correction_dict, correction_dict_rules
    input_path = str(input_path or CFG.pdf_path)
    run_id     = datetime.now().strftime('run_%Y%m%d_%H%M%S')
    t0         = time.time()
    correction_dict_rules = load_correction_dict_v2()
    correction_dict = {k: v.correction if hasattr(v, 'correction') else v
                       for k, v in correction_dict_rules.items()}

    # استئناف من checkpoint
    ckpt = load_checkpoint() if resume else None
    if ckpt and ckpt.get('input_path') == input_path:
        start_page = int(ckpt.get('next_page', start_page or CFG.pages_start))
        LOGGER.info(f'استئناف من الصفحة {start_page}')

    DB.insert_run(run_id, input_path)
    log_event('process_started', {'run_id': run_id, 'input': input_path})

    try:
        pages, page_nums = load_pages(input_path, start_page, end_page)
    except Exception as e:
        log_event('load_failed', {'error': str(e)}, 'error')
        return {'error': str(e)}

    DB.delete_pages(min(page_nums), max(page_nums))
    total_words, conf_acc = 0, []

    with FileLock(CFG.lock_file, timeout=CFG.sync_lock_timeout):
        for idx, (page_img, actual_pg) in enumerate(zip(pages, page_nums)):
            if progress_cb:
                progress_cb(idx+1, len(pages), f'صفحة {actual_pg}/{page_nums[-1]}...')

            # EasyOCR detection
            try:
                detections = easy_reader.readtext(page_img, detail=1)
            except Exception as e:
                detections = []; LOGGER.warning(f'EasyOCR p{actual_pg}: {e}')

            binary, _ = preprocess_image(page_img, adaptive=adaptive)
            boxes = smart_segmentation(page_img, binary, detections)
            del binary
            boxes_info = match_boxes_detections(boxes, detections)

            # ---- تصنيف المحاصيل: EasyOCR واثق vs يحتاج TrOCR ----
            need_trocr_idx, need_trocr_crops = [], []
            easy_results = []
            for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0:
                    easy_results.append(None); continue
                if easy_item is not None:
                    _, txt, conf = easy_item
                    easy_results.append(('easyocr', normalize_text(txt), float(conf)))
                    if float(conf) < CFG.easy_conf_threshold:
                        need_trocr_idx.append(i); need_trocr_crops.append(crop)
                else:
                    easy_results.append(None)
                    need_trocr_idx.append(i); need_trocr_crops.append(crop)

            # ---- Batch TrOCR ----
            trocr_texts = {}
            for b_start in range(0, len(need_trocr_crops), CFG.trocr_batch_size):
                batch_crops = need_trocr_crops[b_start:b_start+CFG.trocr_batch_size]
                texts = trocr_batch_predict(batch_crops)
                for k, txt in enumerate(texts):
                    trocr_texts[need_trocr_idx[b_start+k]] = txt
                del batch_crops
            del need_trocr_crops

            # ---- الدمج والإدراج ----
            for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0: continue
                easy_res = easy_results[i]

                if easy_res and easy_res[2] >= CFG.easy_conf_threshold:
                    raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                elif i in trocr_texts and trocr_texts[i]:
                    raw, conf, src = trocr_texts[i], CFG.trocr_default_conf, 'trocr'
                    if easy_res and easy_res[2] > conf:
                        raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                elif easy_res:
                    raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                else:
                    raw, conf, src = '', 0.0, 'none'

                corrected = apply_corrections_v2(spell_correct(raw), correction_dict_rules)
                if raw.strip() in correction_dict_rules:
                    track_correction_usage(raw.strip(), correction_dict_rules[raw.strip()].correction)
                _, buf = cv2.imencode('.png', crop)
                DB.insert_word(
                    image_data=buf.tobytes(), predicted_text=corrected,
                    raw_text=raw, status='unverified', confidence=conf,
                    model_source=src, x=x, y=y, w=w, h=h,
                    page_num=actual_pg, run_id=run_id)
                total_words += 1; conf_acc.append(conf)
                del buf

            # ---- تنظيف الذاكرة بعد كل صفحة ----
            del page_img, boxes_info, trocr_texts, easy_results
            gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()

            save_checkpoint({'run_id': run_id, 'input_path': input_path,
                             'next_page': actual_pg+1, 'words': total_words,
                             'ts': datetime.now().isoformat()})
            log_event('page_done', {'page': actual_pg, 'boxes': len(boxes),
                                     'words_total': total_words})

    duration = time.time() - t0
    avg_conf  = float(np.mean(conf_acc)) if conf_acc else 0.0
    clear_checkpoint()
    DB.finish_run(run_id, len(page_nums), total_words, avg_conf)

    stats = {
        'run_id': run_id, 'ts': datetime.now().isoformat(),
        'input': input_path, 'pages': len(page_nums),
        'words': total_words, 'avg_confidence': round(avg_conf, 4),
        'duration_sec': round(duration, 2),
    }
    CFG.stats_json.write_text(json.dumps(stats, ensure_ascii=False, indent=2), 'utf-8')

    # runs history
    runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
    runs = pd.concat([runs, pd.DataFrame([{
        'run_id': run_id, 'timestamp': stats['ts'],
        'pages': len(page_nums), 'words': total_words,
        'avg_conf': avg_conf, 'duration_sec': duration, 'status': 'completed',
    }])], ignore_index=True)
    runs.to_csv(CFG.runs_csv, index=False, encoding='utf-8-sig')

    if CFG.auto_export:
        stats['export'] = auto_export(run_id)

    log_event('process_finished', stats)
    print(f'✅ اكتملت | {total_words} كلمة | {duration:.1f}s')
    return stats

print('✅ معالج PDF جاهز')

In [ ]:
def auto_export(run_id: str, output_dir: Path = None) -> dict:
    """CSV + XLSX + نص كامل + JSONL للتدريب"""
    output_dir = output_dir or (CFG.exports_dir / 'auto' / run_id)
    output_dir.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        df_all = pd.read_sql_query(
            'SELECT * FROM handwriting_data ORDER BY page_num,y,x', c)
    df_verified = df_all[df_all['status'].isin(['verified','sentence_corrected'])]
    df_csv = df_all.drop(columns=['image_data'], errors='ignore')
    exported = {}

    # CSV
    p = output_dir / 'all_words.csv'
    df_csv.to_csv(p, index=False, encoding='utf-8-sig')
    exported['csv'] = str(p)

    # XLSX
    px = output_dir / 'all_words.xlsx'
    with pd.ExcelWriter(px, engine='openpyxl') as w:
        df_csv.to_excel(w, sheet_name='All', index=False)
        for pg in sorted(df_csv['page_num'].dropna().unique()):
            df_csv[df_csv['page_num']==pg].to_excel(
                w, sheet_name=f'P{int(pg)}', index=False)
    exported['xlsx'] = str(px)

    # النص الكامل
    lines_out = []
    for pg in sorted(df_all['page_num'].dropna().unique()):
        pw = df_all[df_all['page_num']==pg].sort_values(['y','x'])
        if pw.empty: continue
        curr = [pw.iloc[0].to_dict()]
        lgroups = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= 25: curr.append(row)
            else: lgroups.append(curr); curr = [row]
        lgroups.append(curr)
        for lg in lgroups:
            lang = detect_lang(' '.join(str(w.get('predicted_text','')) for w in lg))
            sl = sorted(lg, key=lambda k: k['x'], reverse=(lang=='ar'))
            lines_out.append(' '.join(str(w.get('predicted_text','')) for w in sl).strip())
    pt = output_dir / 'reconstructed_text.txt'
    pt.write_text('\n'.join(lines_out), 'utf-8')
    exported['text'] = str(pt)

    # JSONL للتدريب
    if not df_verified.empty:
        img_dir = output_dir / 'training_images'
        img_dir.mkdir(exist_ok=True)
        records = []
        for _, row in df_verified.iterrows():
            fname = f"img_{row['image_id']}.png"
            with open(img_dir / fname, 'wb') as f:
                f.write(row['image_data'])
            txt = normalize_text(row['predicted_text'])
            if txt: records.append({'image': fname, 'text': txt})
        pj = output_dir / 'training_data.jsonl'
        with open(pj, 'w', encoding='utf-8') as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        exported['jsonl']    = str(pj)
        exported['samples']  = len(records)

    summary = {
        'run_id': run_id, 'exported_at': datetime.now().isoformat(),
        'total_words': len(df_all), 'verified': len(df_verified),
        'dir': str(output_dir), 'files': exported,
    }
    (output_dir / 'export_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), 'utf-8')
    log_event('auto_export_done', {'dir': str(output_dir)})
    del df_all, df_verified, df_csv
    gc.collect()
    return summary


def create_backup() -> str:
    label  = datetime.now().strftime('%Y%m%d_%H%M%S')
    bk_dir = CFG.backups_dir / f'backup_{label}'
    bk_dir.mkdir(parents=True, exist_ok=True)
    for p in [CFG.db_path, CFG.feedback_csv, CFG.stats_json,
              CFG.correction_dict_path, _LOG_FILE, _EVENTS]:
        if Path(p).exists():
            shutil.copy2(p, bk_dir / Path(p).name)
    log_event('backup_created', {'dir': str(bk_dir)})
    return str(bk_dir)

print('✅ نظام التصدير جاهز')

In [ ]:
def reconstruct_sentences(verified_only=False, y_tolerance=25) -> pd.DataFrame:
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        q = "SELECT * FROM handwriting_data"
        if verified_only:
            q += " WHERE status IN ('verified','sentence_corrected')"
        q += " ORDER BY page_num,y,x"
        words = pd.read_sql_query(q, c)
    if words.empty: return pd.DataFrame()
    out = []
    for pg in words['page_num'].unique():
        pw = words[words['page_num']==pg].sort_values(['y','x'])
        curr = [pw.iloc[0].to_dict()]
        lines = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= y_tolerance: curr.append(row)
            else: lines.append(curr); curr = [row]
        lines.append(curr)
        for line in lines:
            preview = ' '.join(str(w.get('predicted_text','')) for w in line)
            lang    = detect_lang(preview)
            sl = sorted(line, key=lambda k: k['x'], reverse=(lang=='ar'))
            sentence = ' '.join(str(w.get('predicted_text','')) for w in sl).strip()
            out.append({'page': line[0]['page_num'], 'y_anchor': line[0]['y'],
                        'lang': lang, 'text': sentence,
                        'word_ids': [w['image_id'] for w in sl]})
    return pd.DataFrame(out)

print('✅ إعادة بناء الجمل جاهزة')

In [ ]:
def compute_metrics() -> dict:
    try:
        from jiwer import wer, cer
    except ImportError:
        return {'error': 'pip install jiwer'}
    words = DB.get_verified()
    valid = [w for w in words
             if w.get('raw_text','').strip() and w.get('predicted_text','').strip()]
    if len(valid) < 5:
        return {'wer': None, 'cer': None, 'samples': len(valid)}
    refs = [w['raw_text'].strip()       for w in valid]
    hyps = [w['predicted_text'].strip() for w in valid]
    m = {'wer': round(wer(refs, hyps), 4),
         'cer': round(cer(refs, hyps), 4),
         'samples': len(valid),
         'timestamp': datetime.now().isoformat()}
    mdf = pd.DataFrame([m])
    if CFG.metrics_log.exists():
        mdf.to_csv(CFG.metrics_log, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        mdf.to_csv(CFG.metrics_log, index=False, encoding='utf-8-sig')
    return m


def plot_metrics_fig():
    if not CFG.metrics_log.exists(): return None
    df = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
    if df.empty: return None
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['wer'].dropna().values, label='WER', marker='o', color='#E53935')
    ax.plot(df['cer'].dropna().values, label='CER', marker='s', color='#1E88E5')
    ax.set_title('تحسّن النموذج (WER/CER)')
    ax.set_xlabel('جلسة التدريب')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

print('✅ Metrics جاهزة')

In [ ]:
def count_corrections() -> int:
    if not CFG.feedback_csv.exists(): return 0
    return len(pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig'))


def should_train(min_n=None) -> bool:
    return count_corrections() >= (min_n or CFG.al_min_corrections)


def reprocess_low_confidence(limit=None) -> int:
    limit = limit or CFG.al_reprocess_limit
    rows  = DB.get_low_confidence(CFG.low_conf_threshold, limit)
    if not rows: return 0
    done = 0
    for row in tqdm(rows, desc='إعادة معالجة'):
        try:
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            texts  = trocr_batch_predict([img_np])
            new    = texts[0].strip() if texts else ''
            if new: DB.update_word(row['image_id'], predicted_text=new); done += 1
            del img, img_np
        except Exception as e: LOGGER.debug(f'reprocess {row["image_id"]}: {e}')
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    log_event('reprocess_done', {'count': done})
    return done

print('✅ Active Learning جاهز')

In [ ]:
def finetune_trocr_lora(min_samples=None, epochs=None, batch_size=None,
                         lr=None, progress_cb=None) -> dict:
    """
    Fine-tuning مع:
    - TensorBoard للمراقبة (tensorboard --logdir <tb_dir> --port 6006)
    - Albumentations augmentation
    - تنظيف الذاكرة بعد كل epoch
    - إعادة تحميل النموذج تلقائياً
    """
    global trocr_model
    try:
        from peft import get_peft_model, LoraConfig, TaskType
        from torch.optim import AdamW
        from torch.utils.data import Dataset as TDS, DataLoader
        from torch.utils.tensorboard import SummaryWriter
    except ImportError as e:
        return {'error': str(e)}

    min_samples = min_samples or CFG.finetune_min_samples
    epochs      = epochs      or CFG.finetune_epochs
    batch_size  = batch_size  or CFG.finetune_batch_size
    lr          = lr          or CFG.finetune_lr

    verified = DB.get_verified()
    if len(verified) < min_samples:
        return {'error': f'عينات غير كافية: {len(verified)}/{min_samples}'}
    print(f'🚀 تدريب على {len(verified)} عينة | lr={lr} | epochs={epochs}')

    writer = SummaryWriter(
        log_dir=str(CFG.tb_dir / datetime.now().strftime('ft_%Y%m%d_%H%M%S')))
    writer.add_text('config', f'epochs={epochs}, lr={lr}, samples={len(verified)}')

    try:
        import albumentations as A
        _aug = A.Compose([
            A.Rotate(limit=3, p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.GaussNoise(p=0.3),
        ])
        USE_AUG = True
    except ImportError:
        USE_AUG = False

    class HWDataset(TDS):
        def __init__(self, records): self.records = records
        def __len__(self):  return len(self.records)
        def __getitem__(self, idx):
            row = self.records[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = np.array(img)
            if USE_AUG: img_np = _aug(image=img_np)['image']
            pv = trocr_processor(
                images=Image.fromarray(img_np),
                return_tensors='pt').pixel_values.squeeze(0)
            if CFG.use_fp16 and DEVICE.type == 'cuda':
                pv = pv.half()
            text = normalize_text(row['predicted_text'])
            lb = trocr_processor.tokenizer(
                text, return_tensors='pt',
                padding='max_length', max_length=CFG.max_text_length,
                truncation=True).input_ids.squeeze(0)
            lb[lb == trocr_processor.tokenizer.pad_token_id] = -100
            return {'pixel_values': pv, 'labels': lb}

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=CFG.lora_r, lora_alpha=CFG.lora_alpha,
        target_modules=CFG.lora_targets, lora_dropout=CFG.lora_dropout)
    model = get_peft_model(trocr_model, lora_cfg)
    model.train()
    loader    = DataLoader(HWDataset(verified), batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)
    global_step, log = 0, []

    for epoch in range(epochs):
        total_loss = 0.0
        for batch in loader:
            pv = batch['pixel_values'].to(DEVICE)
            lb = batch['labels'].to(DEVICE)
            out = model(pixel_values=pv, labels=lb)
            out.loss.backward()
            optimizer.step(); optimizer.zero_grad()
            loss_val = float(out.loss.item())
            total_loss += loss_val
            writer.add_scalar('Loss/step', loss_val, global_step)
            global_step += 1
            del pv, lb, out

        avg_loss = total_loss / max(1, len(loader))
        writer.add_scalar('Loss/epoch', avg_loss, epoch)
        writer.add_scalar('LR', lr, epoch)
        msg = f'Epoch {epoch+1}/{epochs} | Loss={avg_loss:.4f}'
        log.append(msg); print(msg)
        if progress_cb: progress_cb(epoch+1, epochs, msg)

        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    writer.close()
    lora_p = CFG.lora_path
    model.save_pretrained(str(lora_p))
    trocr_processor.save_pretrained(str(lora_p))
    trocr_model = model
    log_event('finetune_done', {'epochs': epochs, 'samples': len(verified)})
    print(f'✅ محفوظ: {lora_p}')
    print(f'🔍 TensorBoard: tensorboard --logdir {CFG.tb_dir} --port 6006')
    return {'status': 'done', 'log': log, 'lora_path': str(lora_p)}


def active_learning_cycle() -> str:
    global trocr_model
    if not should_train():
        return f'⏳ التصحيحات: {count_corrections()}/{CFG.al_min_corrections}'
    print('🧠 Active Learning...')
    result = finetune_trocr_lora()
    if 'error' in result: return f'❌ {result["error"]}'
    if CFG.lora_path.exists():
        try:
            from peft import PeftModel
            trocr_model = PeftModel.from_pretrained(
                trocr_model, str(CFG.lora_path)).to(DEVICE)
        except Exception as e: LOGGER.warning(f'reload lora: {e}')
    reprocessed = reprocess_low_confidence()
    m = compute_metrics()
    log_event('al_cycle_done', {'reprocessed': reprocessed, 'metrics': m})
    return (f'✅ دورة AL اكتملت\n'
            f'إعادة معالجة: {reprocessed} كلمة\n'
            f'WER: {m.get("wer")} | CER: {m.get("cer")}')


def upload_to_hf(repo_id=None, hf_token=None) -> str:
    from huggingface_hub import HfApi, login
    repo_id  = repo_id  or CFG.hf_dataset_repo
    hf_token = hf_token or CFG.hf_token or _HF_TOKEN
    if not repo_id:  return '❌ يجب تحديد repo_id'
    if not hf_token: return '❌ يجب توفير HF_TOKEN'
    stats = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    run_id = stats.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d")}')
    export_dir = CFG.exports_dir / 'auto' / run_id
    if not export_dir.exists(): auto_export(run_id)
    try:
        login(token=hf_token)
        api = HfApi()
        api.create_repo(repo_id=repo_id, repo_type='dataset', exist_ok=True)
        api.upload_folder(
            folder_path=str(export_dir), repo_id=repo_id, repo_type='dataset',
            commit_message=f'Update {datetime.now().strftime("%Y-%m-%d")}')
        url = f'https://huggingface.co/datasets/{repo_id}'
        log_event('hf_upload', {'url': url})
        return f'✅ تم الرفع: {url}'
    except Exception as e: return f'❌ {e}'

print('✅ Fine-tuning + Active Learning + HF جاهزان')

In [ ]:
def generate_study_guide(output_dir=None, include_mermaid=True,
                          include_flashcards=True, title=None) -> str:
    """
    توليد مرجع دراسي شامل من البيانات المستخرجة:
    - جداول المصطلحات (EN↔AR)
    - الملاحظات المُعاد بناؤها
    - مخطط Mermaid للعلاقات
    - بطاقات تعليمية (Flashcards)
    - تصدير HTML للطباعة
    """
    output_dir = Path(output_dir or CFG.study_guide_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    title = title or 'مرجع دراسة — مستخرج من الملاحظات اليدوية'

    words = DB.get_all()
    if not words:
        return '⚠️ لا توجد بيانات. شغّل المعالجة أولاً.'

    df = pd.DataFrame(words)
    pages = sorted(df['page_num'].dropna().unique())
    guide = [f'# {title}', f'تاريخ: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
             f'صفحات: {len(pages)}', '---']

    vocab_pairs = []   # لـ Mermaid + Flashcards

    for pg in pages:
        pw = df[df['page_num']==pg].sort_values(['y','x'])
        if pw.empty: continue
        guide.append(f'\n## صفحة {int(pg)}\n')

        # --- جدول المصطلحات ---
        table_rows = []
        curr = [pw.iloc[0].to_dict()]
        lgroups = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= 25: curr.append(row)
            else: lgroups.append(curr); curr = [row]
        lgroups.append(curr)

        for lg in lgroups:
            en_w = [str(w.get('predicted_text','')).strip()
                    for w in lg
                    if str(w.get('predicted_text','')).strip()
                    and all(ord(c)<128 for c in str(w.get('predicted_text','')).replace(' ',''))]
            ar_w = [str(w.get('predicted_text','')).strip()
                    for w in lg
                    if str(w.get('predicted_text','')).strip()
                    and any('\u0600'<=c<='\u06FF' for c in str(w.get('predicted_text','')))]
            if en_w or ar_w:
                table_rows.append({'EN': ' | '.join(en_w), 'AR': ' | '.join(ar_w)})
                if en_w and ar_w:
                    vocab_pairs.append({'en': ' '.join(en_w), 'ar': ' '.join(ar_w), 'page': int(pg)})

        if table_rows:
            guide.append('### جداول المصطلحات\n')
            guide.append('| إنجليزي | عربي |')
            guide.append('|---------|------|')
            for r in table_rows:
                guide.append(f"| {r['EN']} | {r['AR']} |")
            guide.append('')

        # --- الجمل ---
        sent_lines = []
        for lg in lgroups:
            texts = [str(w.get('predicted_text','')).strip() for w in lg if w.get('predicted_text')]
            if not texts: continue
            txt = ' '.join(texts)
            lang = detect_lang(txt)
            sl = sorted(lg, key=lambda k: k['x'], reverse=(lang=='ar'))
            sent = ' '.join(str(w.get('predicted_text','')) for w in sl).strip()
            if sent: sent_lines.append(f'- {sent}')
        if sent_lines:
            guide.append('### الملاحظات\n')
            guide.extend(sent_lines)
            guide.append('')

    # --- Mermaid ---
    mermaid_code = ''
    if include_mermaid and vocab_pairs:
        guide.append('\n---\n## خريطة المفردات\n')
        lines = ['mindmap', '  root((المصطلحات))']
        for v in vocab_pairs[:40]:
            en_id = v['en'].replace(' ','_')[:20]
            ar_id = v['ar'].replace(' ','_')[:20]
            lines.extend([f'    {en_id}[{v["en"]}]', f'      {ar_id}[{v["ar"]}]'])
        mermaid_code = '\n'.join(lines)
        guide.append(f'```mermaid\n{mermaid_code}\n```\n')
        (output_dir / 'vocab_diagram.mmd').write_text(mermaid_code, 'utf-8')

    # --- Flashcards ---
    if include_flashcards and vocab_pairs:
        guide.append('\n---\n## البطاقات التعليمية\n')
        import csv
        cards_path = output_dir / 'flashcards_anki.csv'
        with open(cards_path, 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.writer(f, delimiter=';')
            w.writerow(['front', 'back', 'tags'])
            for v in vocab_pairs:
                w.writerow([v['en'], v['ar'], f"page_{v['page']} EN-AR"])
                w.writerow([v['ar'], v['en'], f"page_{v['page']} AR-EN"])
        guide.append(f'تم توليد {len(vocab_pairs)*2} بطاقة → `{cards_path.name}`\n')

    content = '\n'.join(guide)

    # --- حفظ ---
    md_path = output_dir / 'study_guide.md'
    md_path.write_text(content, 'utf-8')

    # HTML للطباعة
    html_path = output_dir / 'study_guide.html'
    _export_html(content, html_path, title)

    print(f'✅ المرجع الدراسي في: {output_dir}')
    print(f'   - Markdown: {md_path.name}')
    print(f'   - HTML: {html_path.name}')
    if include_flashcards and vocab_pairs:
        print(f'   - Anki CSV: flashcards_anki.csv')
    return content


def _export_html(md_content: str, output_path: Path, title: str):
    """تحويل Markdown بسيط إلى HTML أنيق للطباعة"""
    html_lines = [f'''<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head><meta charset="UTF-8"><title>{title}</title>
<style>
body{{font-family:\'Amiri\',\'Segoe UI\',sans-serif;max-width:900px;margin:0 auto;
padding:30px;line-height:1.8;color:#1a1a2e}}
h1{{text-align:center;color:#16213e;border-bottom:3px solid #0f3460;padding-bottom:15px}}
h2{{color:#0f3460;border-right:4px solid #e94560;padding-right:15px}}
h3{{color:#533483}}
table{{border-collapse:collapse;width:100%;margin:15px 0}}
th,td{{border:1px solid #ddd;padding:10px;text-align:right}}
th{{background:#0f3460;color:white}}
tr:nth-child(even){{background:#f8f9fa}}
@media print{{table{{page-break-inside:avoid}}}}
</style></head><body>''']
    in_table = False
    for line in md_content.splitlines():
        s = line.strip()
        if not s: continue
        if s.startswith('# '): html_lines.append(f'<h1>{s[2:]}</h1>')
        elif s.startswith('## '): html_lines.append(f'<h2>{s[3:]}</h2>')
        elif s.startswith('### '): html_lines.append(f'<h3>{s[4:]}</h3>')
        elif s.startswith('---'): html_lines.append('<hr>')
        elif s.startswith('|'):
            cells = [c.strip() for c in s.split('|') if c.strip()]
            if all(set(c) <= {'-'} for c in cells): continue
            if not in_table: html_lines.append('<table>'); in_table = True
            html_lines.append('<tr>' + ''.join(f'<td>{c}</td>' for c in cells) + '</tr>')
        else:
            if in_table: html_lines.append('</table>'); in_table = False
            if s.startswith('- '): html_lines.append(f'<li>{s[2:]}</li>')
            elif s.startswith('```'): pass
            else: html_lines.append(f'<p>{s}</p>')
    if in_table: html_lines.append('</table>')
    html_lines.append('</body></html>')
    output_path.write_text('\n'.join(html_lines), 'utf-8')

print('✅ Study Guide جاهز')

In [ ]:
_review_state = {'df': pd.DataFrame(), 'idx': 0}
_sent_state   = {'df': pd.DataFrame(), 'idx': 0}


def _word_row():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return None, '', '', 'لا توجد كلمات للمراجعة', 0.0, '0/0'
    row  = df.iloc[idx]
    pil  = Image.open(io.BytesIO(bytes(row['image_data'])))
    conf = float(row['confidence'] or 0)
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page_num']} "
            f"| {row['model_source']} | id={row['image_id']}")
    return (pil, str(row['predicted_text'] or ''),
            str(row['raw_text'] or ''), info, conf, f'{idx+1}/{len(df)}')


def load_word_review():
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' "
            "ORDER BY confidence ASC, image_id ASC", c)
    _review_state['df'] = df; _review_state['idx'] = 0
    return _word_row()


def word_confirm(corrected_text: str):
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty: return _word_row()
    row  = df.iloc[idx]; rid = int(row['image_id'])
    orig = normalize_text(row['predicted_text'])
    corr = normalize_text(corrected_text)
    DB.update_word(rid, predicted_text=corr, status='verified')
    DB.log_review(rid, orig, corr, 'verified') # Use 'verified' action for consistency
    if orig != corr: append_feedback(rid, orig, corr, 'verified')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_delete():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty: return _word_row()
    rid = int(df.iloc[idx]['image_id'])
    DB.delete_word(rid); DB.log_review(rid, '', '', 'delete')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_prev():
    if not _review_state['df'].empty:
        _review_state['idx'] = max(0, _review_state['idx']-1)
    return _word_row()


def word_next():
    df = _review_state['df']
    if not df.empty:
        _review_state['idx'] = min(len(df)-1, _review_state['idx']+1)
    return _word_row()


def _sent_row():
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty: return '', 'لا توجد جمل', '0/0'
    row = df.iloc[idx]
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page']} "
            f"| {row['lang']} | {len(row['word_ids'])} كلمة")
    return row['text'], info, f'{idx+1}/{len(df)}'


def load_sent_review():
    _sent_state['df'] = reconstruct_sentences(verified_only=False)
    _sent_state['idx'] = 0
    return _sent_row()


def sent_save(corrected: str):
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty: return _sent_row()
    row  = df.iloc[idx]
    orig = normalize_text(row['text'])
    corr = normalize_text(corrected)
    ts   = datetime.now().isoformat()
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        for wid in row['word_ids']:
            c.execute("UPDATE handwriting_data SET status='sentence_corrected',"
                      "updated_at=? WHERE image_id=?", (ts, int(wid)))
        c.commit()
    append_feedback(None, orig, corr, f"sent_p{row['page']}")
    _sent_state['idx'] = min(idx+1, max(0, len(df)-1))
    return _sent_row()


def sent_prev():
    if not _sent_state['df'].empty:
        _sent_state['idx'] = max(0, _sent_state['idx']-1)
    return _sent_row()


def sent_next():
    df = _sent_state['df']
    if not df.empty:
        _sent_state['idx'] = min(len(df)-1, _sent_state['idx']+1)
    return _sent_row()


def build_dashboard_fig():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    counts = DB.count_by_status()
    if counts:
        axes[0].bar(list(counts.keys()), list(counts.values()),
                    color=['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0'])
        axes[0].set_title('توزيع الحالات')
        axes[0].tick_params(axis='x', rotation=30)
    else:
        axes[0].text(0.5, 0.5, 'لا بيانات', ha='center')
        axes[0].set_title('توزيع الحالات')

    if CFG.runs_csv.exists():
        runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
        if not runs.empty:
            vals = pd.to_numeric(runs['words'], errors='coerce').fillna(0)
            axes[1].plot(vals.values, marker='o', color='#2196F3')
            axes[1].set_title('كلمات لكل تشغيل')
        else:
            axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
            axes[1].set_title('كلمات لكل تشغيل')
    else:
        axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
        axes[1].set_title('كلمات لكل تشغيل')

    if CFG.metrics_log.exists():
        mdf = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
        if not mdf.empty:
            axes[2].plot(mdf['wer'].dropna().values, label='WER',
                         color='#E53935', marker='o')
            axes[2].plot(mdf['cer'].dropna().values, label='CER',
                         color='#1E88E5', marker='s')
            axes[2].set_title('WER / CER'); axes[2].legend()
            axes[2].set_ylim(0, 1)
        else:
            axes[2].text(0.5, 0.5, 'لا مقاييس', ha='center')
            axes[2].set_title('WER / CER')
    else:
        axes[2].text(0.5, 0.5, 'لا مقاييس بعد', ha='center')
        axes[2].set_title('WER / CER')

    plt.tight_layout()
    return fig


def get_dashboard_text() -> str:
    lines = ['## 📊 ملخص المشروع']
    counts = DB.count_by_status()
    lines.append(f"**إجمالي الكلمات:** {sum(counts.values())}")
    for k, v in counts.items(): lines.append(f'- {k}: **{v}**')
    if CFG.stats_json.exists():
        s = json.loads(CFG.stats_json.read_text())
        lines.append(f"\n**آخر تشغيل:** `{s.get('run_id','—')}`  \n"
                     f"صفحات: {s.get('pages')} | كلمات: {s.get('words')} "
                     f"| ثقة: {s.get('avg_confidence')} | مدة: {s.get('duration_sec')}s")

    # معلومات الشبكة (مفيدة للوصول من الجوال)
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(('8.8.8.8', 80)); ip = s.getsockname()[0]; s.close()
        lines.append(f"\n**الوصول من الجوال:** `http://{ip}:{CFG.gradio_port}`")
    except Exception:
        pass

    tail = []
    if _EVENTS.exists():
        with open(_EVENTS, encoding='utf-8', errors='ignore') as f:
            tail = [l.rstrip() for l in f.readlines()[-8:]]
    if tail:
        lines.append('\n**آخر أحداث:**\n```\n' + '\n'.join(tail) + '\n```')
    return '\n'.join(lines)


# --- Gradio wrappers ---
def do_process(inp, sp, ep, resume, adaptive, progress=gr.Progress()):
    ep_val = int(ep) if ep and str(ep).strip() else None
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    try:
        stats = process_document(inp, int(sp), ep_val, resume, adaptive, cb)
        return f"✅ اكتملت\n```json\n{json.dumps(stats, ensure_ascii=False, indent=2)}\n```"
    except Exception as e: return f'❌ {e}'

def do_export():
    s = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    rid = s.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
    summary = auto_export(rid)
    return f"✅\n```json\n{json.dumps(summary, ensure_ascii=False, indent=2)}\n```"

def do_backup(): return f'✅ `{create_backup()}`'

def do_metrics():
    m = compute_metrics()
    if 'error' in m: return m['error'], None
    return (f"**WER:** {m['wer']} | **CER:** {m['cer']} | **عينات:** {m['samples']}",
            build_dashboard_fig())

def do_finetune(min_s, ep, bs, lr, progress=gr.Progress()):
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    result = finetune_trocr_lora(int(min_s), int(ep), int(bs), float(lr), cb)
    if 'error' in result: return f"❌ {result['error']}"
    return '✅ اكتمل\n' + '\n'.join(result.get('log', []))

def do_al():      return active_learning_cycle()
def do_upload(r, t): return upload_to_hf(r or None, t or None)

def do_migrate():
    report = MIGRATOR.scan()
    n = report['total']
    if n == 0:
        return '⚠️ لم يتم العثور على مشاريع قديمة.'
    stats = MIGRATOR.migrate()
    return (f'✅ الترحيل اكتمل:\n'
            f'- سجلات DB: {stats["db_records"]}\n'
            f'- تصحيحات: {stats["feedback"]}\n'
            f'- قاموس: {stats["dict_entries"]} إدخال')

def do_study_guide():
    content = generate_study_guide()
    if content.startswith('⚠️'): return content
    return f'✅ تم حفظ المرجع في: {CFG.study_guide_dir}\n\n{content[:500]}...'


def word_copy_raw(raw_text: str):
    """ينسخ النص الخام إلى حقل النص المعدّل"""
    return raw_text

def word_undo():
    """يتراجع عن آخر تصحيح مع تحديث القاموس تلقائياً"""
    global correction_dict, correction_dict_rules
    last_event = DB.get_last_review_event()
    if not last_event or last_event.get('image_id') is None:
        log_event('undo_failed', {'reason': 'لا يوجد حدث'}, 'info')
        return _word_row()
    event_id = last_event['id']
    image_id = last_event['image_id']
    original_text = normalize_text(last_event.get('original_text', ''))
    corrected_text = normalize_text(last_event.get('corrected_text', ''))

    DB.update_word(image_id, predicted_text=original_text, status='unverified')

    if CFG.feedback_csv.exists():
        try:
            df_feedback = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
            mask = ((df_feedback['image_id'] == image_id) &
                    (df_feedback['original_text'] == original_text) &
                    (df_feedback['corrected_text'] == corrected_text))
            df_filtered = df_feedback[~mask]
            df_filtered.to_csv(CFG.feedback_csv, index=False, encoding='utf-8-sig')
        except Exception:
            pass

    DB.delete_review_event(event_id)

    # تحديث القاموس
    if CFG.correction_dict_path.exists():
        try:
            with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
                raw_dict = json.load(f)
            if original_text in raw_dict:
                rule = CorrectionRule.from_dict(raw_dict[original_text], original_text)
                rule.votes = max(0, rule.votes - 1)
                if rule.votes == 0:
                    _archive_correction_rule(rule, 'undo')
                    del raw_dict[original_text]
                else:
                    raw_dict[original_text] = rule.to_dict()
                with open(CFG.correction_dict_path, 'w', encoding='utf-8') as f:
                    json.dump(raw_dict, f, ensure_ascii=False, indent=2)
            correction_dict_rules = load_correction_dict_v2()
            correction_dict = {k: v.correction if hasattr(v, 'correction') else v
                               for k, v in correction_dict_rules.items()}
        except Exception:
            pass

    log_event('undo_word', {'image_id': image_id})
    return load_word_review()

        # إعادةpredicted_text إلى original_text في جدول DB الرئيسي
        DB.update_word(image_id, predicted_text=original_text, status='unverified')
        log_event('undo_word_correction', {'image_id': image_id, 'reverted_to': original_text})

        # إزالة الإدخال المقابل من feedback_csv إذا كان موجودًا
        if CFG.feedback_csv.exists():
            df_feedback = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
            df_feedback_filtered = df_feedback[
                ~((df_feedback['image_id'] == image_id) &
                  (df_feedback['original_text'] == original_text) &
                  (df_feedback['corrected_text'] == corrected_text_at_undo) &
                  (df_feedback['status'] == 'verified'))
            ].copy()
            df_feedback_filtered.to_csv(CFG.feedback_csv, index=False, encoding='utf-8-sig')
            log_event('feedback_csv_entry_removed', {'image_id': image_id, 'original': original_text, 'corrected': corrected_text_at_undo})

        # حذف سجل حدث المراجعة نفسه
        DB.delete_review_event(event_id)

        # إعادة بناء correction_dict (لأنه يعتمد على feedback_csv)
        global correction_dict
        correction_dict = build_correction_dict()

        # إعادة تحميل واجهة مستخدم مراجعة الكلمات
        return load_word_review()
    else:
        log_event('undo_failed', {'reason': 'لم يتم العثور على حدث تصحيح كلمة فردية للتراجع عنه.'}, 'info')
        return _word_row()



def _archive_correction_rule(rule: CorrectionRule, reason: str = ""):
    """أرشفة قاعدة تصحيح محذوفة"""
    try:
        archive_path = CFG.artifacts_dir / 'corrections_archive.jsonl'
        entry = {'archived_at': datetime.now().isoformat(), 'reason': reason, **rule.to_dict()}
        with open(archive_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    except Exception as e:
        LOGGER.debug(f'archive_rule: {e}')


# ========== حالة مراجعة القاموس ==========
_dict_audit_state = {'rules': [], 'idx': 0, 'priority': 'flagged'}

def load_dict_audit(priority='flagged', limit=20):
    rules = get_dictionary_audit_queue(priority, limit)
    _dict_audit_state.update({'rules': rules, 'idx': 0, 'priority': priority})
    return _render_dict_rule(0) if rules else ("", "", "", "", "", "", "0/0")

def get_dictionary_audit_queue(priority, limit):
    if not CFG.correction_dict_path.exists():
        return []
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        rules = [CorrectionRule.from_dict(v, k) for k, v in raw.items()]
    except Exception:
        return []
    now = datetime.now()
    if priority == 'flagged':
        candidates = [r for r in rules if r.flagged]
    elif priority == 'low_confidence':
        candidates = [r for r in rules if r.confidence < 0.7]
    elif priority == 'recent':
        candidates = [r for r in rules if r.last_reviewed is None and (now - datetime.fromisoformat(r.first_seen)).days <= 7]
    elif priority == 'high_usage':
        candidates = [r for r in rules if r.last_reviewed and r.usage_count > 50 and (now - datetime.fromisoformat(r.last_reviewed)).days > 30]
    else:
        candidates = rules
    sorted_rules = sorted(candidates, key=lambda r: (r.confidence, -r.usage_count))
    return sorted_rules[:limit]

def _render_dict_rule(idx):
    rules = _dict_audit_state['rules']
    if not rules or idx < 0 or idx >= len(rules):
        return "", "", "", "", "", "", "0/0"
    rule = rules[idx]
    status = calculate_rule_indicator(rule)
    indicator_html = (
        f'<div style="padding:8px 12px; background:{status["color"]}15; '
        f'border-left:4px solid {status["color"]}; border-radius:6px; '
        f'margin-bottom:8px; font-weight:bold; color:{status["color"]}">'
        f'{status["indicator"]} — {status["summary"]}</div>'
    )
    meta_info = indicator_html + (
        f"<br><small>📅 أضيفت: {rule.first_seen[:10]} | "
        f"👤 المراجع: {rule.reviewer or '—'} | 📝: {rule.notes or '—'}</small>"
    )
    contexts_text = ""
    if rule.contexts:
        contexts_text = "**السياقات الأخيرة:**\n" + "\n".join(
            f"- `{c.get('predicted_text', '')}` (صفحة {c.get('page_num', '?')})"
            for c in rule.contexts[:3]
        )
    return rule.original, rule.correction, meta_info, contexts_text, status["indicator"], "", f"{idx+1}/{len(rules)}"

def dict_audit_prev():
    _dict_audit_state['idx'] = max(0, _dict_audit_state['idx'] - 1)
    return _render_dict_rule(_dict_audit_state['idx'])

def dict_audit_next():
    _dict_audit_state['idx'] = min(len(_dict_audit_state['rules']) - 1, _dict_audit_state['idx'] + 1)
    return _render_dict_rule(_dict_audit_state['idx'])

def dict_audit_edit(new_correction):
    global correction_dict, correction_dict_rules
    rules = _dict_audit_state['rules']
    idx = _dict_audit_state['idx']
    if not rules or idx < 0:
        return _render_dict_rule(idx)
    rule = rules[idx]
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        if rule.original in raw:
            old_rule = CorrectionRule.from_dict(raw[rule.original], rule.original)
            old_rule.correction = new_correction
            old_rule.confidence = 1.0
            old_rule.flagged = False
            old_rule.last_reviewed = datetime.now().isoformat()
            old_rule.reviewer = 'user'
            raw[rule.original] = old_rule.to_dict()
            with open(CFG.correction_dict_path, 'w', encoding='utf-8') as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)
            correction_dict_rules = load_correction_dict_v2()
            correction_dict = {k: v.correction if hasattr(v, 'correction') else v
                               for k, v in correction_dict_rules.items()}
            rules[idx] = old_rule
    except Exception:
        pass
    return _render_dict_rule(idx)

def dict_audit_approve():
    rules = _dict_audit_state['rules']
    idx = _dict_audit_state['idx']
    if not rules or idx < 0:
        return _render_dict_rule(idx)
    rule = rules[idx]
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        if rule.original in raw:
            old_rule = CorrectionRule.from_dict(raw[rule.original], rule.original)
            old_rule.flagged = False
            old_rule.last_reviewed = datetime.now().isoformat()
            old_rule.reviewer = 'user'
            old_rule.notes = old_rule.notes or 'تمت المراجعة'
            raw[rule.original] = old_rule.to_dict()
            with open(CFG.correction_dict_path, 'w', encoding='utf-8') as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)
    except Exception:
        pass
    return _render_dict_rule(idx)

def dict_audit_reject():
    global correction_dict, correction_dict_rules
    rules = _dict_audit_state['rules']
    idx = _dict_audit_state['idx']
    if not rules or idx < 0:
        return _render_dict_rule(idx)
    rule = rules[idx]
    try:
        with open(CFG.correction_dict_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        if rule.original in raw:
            _archive_correction_rule(CorrectionRule.from_dict(raw[rule.original], rule.original), 'rejected')
            del raw[rule.original]
            with open(CFG.correction_dict_path, 'w', encoding='utf-8') as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)
            correction_dict_rules = load_correction_dict_v2()
            correction_dict = {k: v.correction if hasattr(v, 'correction') else v
                               for k, v in correction_dict_rules.items()}
        rules.pop(idx)
        _dict_audit_state['idx'] = min(idx, len(rules)-1)
    except Exception:
        pass
    return _render_dict_rule(_dict_audit_state['idx'])

print('✅ Gradio helpers جاهزة')

In [ ]:
def build_app():
    with gr.Blocks(
        title='Arabic Handwriting OCR — v5 Enhanced'
    ) as app:

        gr.Markdown(
            '# 🖋️ Arabic Handwriting OCR — v5 Enhanced\n'
            '> TrOCR Batch + Beam Search + LoRA | Active Learning | WER/CER '
            '| Study Guide | Enhanced Dictionary | FileLock Multi-Device')

        # ==================== TAB 1: المعالجة ====================
        with gr.Tab('⚙️ المعالجة'):
            gr.Markdown('### معالجة PDF أو صورة')
            with gr.Row():
                inp_path   = gr.Textbox(label='مسار الملف', value=CFG.pdf_path)
                start_page = gr.Number(label='من الصفحة', value=CFG.pages_start, precision=0)
                end_page   = gr.Textbox(label='إلى الصفحة (فارغ=كل الصفحات)',
                                        value=str(CFG.pages_end))
            with gr.Row():
                resume_cb   = gr.Checkbox(label='استئناف من Checkpoint', value=True)
                adaptive_cb = gr.Checkbox(label='Adaptive Threshold', value=False)
            run_btn = gr.Button('🚀 ابدأ المعالجة', variant='primary')
            run_out = gr.Markdown()
            run_btn.click(do_process,
                          [inp_path, start_page, end_page, resume_cb, adaptive_cb], run_out)

        # ==================== TAB 2: مراجعة الكلمات ====================
        with gr.Tab('🔍 مراجعة الكلمات'):
            load_w_btn  = gr.Button('📥 تحميل الكلمات')
            word_info   = gr.Markdown('—')
            word_prog   = gr.Textbox(label='التقدم', interactive=False)
            word_img    = gr.Image(label='الصورة', type='pil', height=160)
            word_raw    = gr.Textbox(label='النص الخام', interactive=False)
            # إضافة زر نسخ النص الخام
            copy_raw_btn = gr.Button('📄 نسخ الخام', variant='secondary', size='sm')
            word_edit   = gr.Textbox(label='النص المعدّل ✏️')
            conf_slider = gr.Slider(0, 1, label='الثقة', interactive=False)
            with gr.Row():
                prev_w = gr.Button('⬅ السابق')
                undo_w = gr.Button('↩️ تراجع', variant='secondary') # زر التراجع الجديد
                conf_w = gr.Button('✅ تأكيد', variant='primary')
                del_w  = gr.Button('🗑 حذف', variant='stop')
                next_w = gr.Button('التالي ➡')
            _wo = [word_img, word_edit, word_raw, word_info, conf_slider, word_prog]
            load_w_btn.click(load_word_review, outputs=_wo)
            copy_raw_btn.click(word_copy_raw, inputs=[word_raw], outputs=[word_edit])
            conf_w.click(word_confirm, [word_edit], _wo)
            del_w.click(word_delete, outputs=_wo)
            prev_w.click(word_prev, outputs=_wo)
            next_w.click(word_next, outputs=_wo)
            undo_w.click(word_undo, outputs=_wo) # ربط زر التراجع

        # ==================== TAB 3: مراجعة الجمل ====================
        with gr.Tab('📝 مراجعة الجمل'):
            load_s_btn = gr.Button('📥 تحميل الجمل')
            sent_info  = gr.Markdown('—')
            sent_prog  = gr.Textbox(label='التقدم', interactive=False)
            sent_edit  = gr.Textbox(label='الجملة ✏️', lines=3)
            with gr.Row():
                prev_s = gr.Button('⬅ السابق')
                save_s = gr.Button('✅ حفظ', variant='primary')
                next_s = gr.Button('التالي ➡')
            _so = [sent_edit, sent_info, sent_prog]
            load_s_btn.click(load_sent_review, outputs=_so)
            save_s.click(sent_save, [sent_edit], _so)
            prev_s.click(sent_prev, outputs=_so)
            next_s.click(sent_next, outputs=_so)

        # ==================== TAB 4: Dashboard ====================
        with gr.Tab('📊 Dashboard'):
            refresh_btn = gr.Button('🔄 تحديث')
            dash_text   = gr.Markdown()
            dash_plot   = gr.Plot()
            with gr.Row():
                export_btn  = gr.Button('📤 تصدير', variant='secondary')
                backup_btn  = gr.Button('💾 نسخة', variant='secondary')
                metrics_btn = gr.Button('📐 WER/CER', variant='secondary')
            export_out  = gr.Markdown()
            backup_out  = gr.Markdown()
            metrics_out = gr.Markdown()
            metrics_plt = gr.Plot()
            refresh_btn.click(
                lambda: (get_dashboard_text(), build_dashboard_fig()),
                outputs=[dash_text, dash_plot])
            export_btn.click(do_export, outputs=export_out)
            backup_btn.click(do_backup, outputs=backup_out)
            metrics_btn.click(do_metrics, outputs=[metrics_out, metrics_plt])
            app.load(lambda: (get_dashboard_text(), build_dashboard_fig()),
                     outputs=[dash_text, dash_plot])

        # ==================== TAB 5: Fine-tuning ====================
        with gr.Tab('🧠 Fine-tuning & AL'):
            gr.Markdown('### LoRA Fine-tuning مع TensorBoard + Augmentation')
            with gr.Row():
                ft_min = gr.Number(label='حد أدنى للعينات',
                                   value=CFG.finetune_min_samples, precision=0)
                ft_ep  = gr.Number(label='Epochs',
                                   value=CFG.finetune_epochs, precision=0)
                ft_bs  = gr.Number(label='Batch size',
                                   value=CFG.finetune_batch_size, precision=0)
                ft_lr  = gr.Number(label='Learning rate', value=CFG.finetune_lr)
            ft_btn = gr.Button('🔥 ابدأ Fine-tuning', variant='primary')
            ft_out = gr.Markdown()
            ft_btn.click(do_finetune, [ft_min, ft_ep, ft_bs, ft_lr], ft_out)
            gr.Markdown('---\n### 🔁 Active Learning تلقائي')
            gr.Markdown(f'الحد الأدنى: **{CFG.al_min_corrections}** تصحيح')
            al_btn = gr.Button('🧠 دورة Active Learning', variant='secondary')
            al_out = gr.Markdown()
            al_btn.click(do_al, outputs=al_out)

        # ==================== TAB 6: Study Guide ====================
        with gr.Tab('📚 دليل الدراسة'):
            gr.Markdown('### توليد مرجع دراسي من ملاحظاتك اليدوية')
            gr.Markdown(
                'يُولّد: جداول المصطلحات (EN↔AR) + جمل مُعادة البناء + '
                'مخطط Mermaid + بطاقات Anki + HTML للطباعة')
            sg_btn = gr.Button('📖 توليد المرجع الدراسي', variant='primary')
            sg_out = gr.Markdown()
            sg_btn.click(do_study_guide, outputs=sg_out)

        # ==================== TAB 7: الإعداد والترحيل ====================
        with gr.Tab('⚙️ الإعداد'):
            gr.Markdown('### ترحيل البيانات من النسخ القديمة')
            gr.Markdown('يبحث تلقائياً في: Handwriting_Dataset, '
                        'Handwritten_OCR_Integrated, Handwritten_OCR_Pro')
            migrate_btn = gr.Button('🔄 بدء الترحيل', variant='secondary')
            migrate_out = gr.Markdown()
            migrate_btn.click(do_migrate, outputs=migrate_out)

            gr.Markdown('---\n### رفع إلى HuggingFace Hub')
            hf_repo  = gr.Textbox(label='Repo ID', placeholder='username/dataset',
                                   value=CFG.hf_dataset_repo)
            hf_token = gr.Textbox(label='HF Token', type='password', value='')
            hf_btn   = gr.Button('🚀 رفع إلى Hub', variant='primary')
            hf_out   = gr.Markdown()
            hf_btn.click(do_upload, [hf_repo, hf_token], hf_out)

        # ==================== TAB 8: مراجعة قاموس التصحيح ====================
        with gr.Tab('📚 مراجعة القاموس'):
            gr.Markdown('### مراجعة وتدقيق قواعد التصحيح\n'
                        '> المؤشر البصري يلخص الثقة، التكرار، وزمن المراجعة')
            with gr.Row():
                priority_dropdown = gr.Dropdown(
                    choices=[('قواعد تحتاج انتباه', 'flagged'),
                             ('ثقة منخفضة', 'low_confidence'),
                             ('أضيفت مؤخراً', 'recent'),
                             ('استخدام عالي', 'high_usage'),
                             ('الكل', 'all')],
                    label='أولوية العرض', value='flagged')
                limit_slider = gr.Slider(10, 100, 20, step=10, label='عدد القواعد')
                load_audit_btn = gr.Button('🔄 تحميل القائمة', variant='secondary')
            with gr.Accordion("⚙️ ضبط العتبات والمعايرة الإحصائية", open=False):
                with gr.Row():
                    th_conf_low = gr.Number(label="حد الثقة المنخفض (🔴)", value=CFG.dict_thresholds["conf_low"])
                    th_conf_mid = gr.Number(label="حد الثقة المتوسط (🟡)", value=CFG.dict_thresholds["conf_mid"])
                    th_usage_high = gr.Number(label="استخدام عالٍ حرج", value=CFG.dict_thresholds["usage_high"])
                    th_usage_mid = gr.Number(label="استخدام متوسط", value=CFG.dict_thresholds["usage_mid"])
                with gr.Row():
                    cal_method = gr.Dropdown(
                        choices=[("النسب المئوية", "percentile"), ("الانحراف المعياري", "std_dev")],
                        value=CFG.dict_thresholds["calibrate_method"], label="طريقة المعايرة")
                    cal_btn = gr.Button("📐 إعادة معايرة تلقائية", variant="secondary")
                    save_btn = gr.Button("💾 حفظ العتبات يدوياً", variant="primary")
                cal_out = gr.Markdown()

                def _save_thresholds(cl, cm, uh, um, method):
                    CFG.dict_thresholds.update({"conf_low": float(cl), "conf_mid": float(cm),
                                                "usage_high": float(uh), "usage_mid": float(um),
                                                "calibrate_method": method})
                    return "✅ تم حفظ العتبات في الذاكرة"

                def _run_calibration(method):
                    res = auto_calibrate_dict_thresholds(method)
                    if "error" in res:
                        return (0, 0, 0, 0, f"❌ {res['error']}")
                    t = res["thresholds"]
                    return (t["conf_low"], t["conf_mid"], t["usage_high"], t["usage_mid"],
                            f"✅ معايرة ناجحة ({res['method']}) على {res['samples']} قاعدة.\n"
                            f"الثقة: {t['conf_low']}/{t['conf_mid']} | الاستخدام: {t['usage_high']}/{t['usage_mid']}")

                cal_btn.click(_run_calibration, inputs=[cal_method],
                              outputs=[th_conf_low, th_conf_mid, th_usage_high, th_usage_mid, cal_out])
                save_btn.click(_save_thresholds,
                               inputs=[th_conf_low, th_conf_mid, th_usage_high, th_usage_mid, cal_method],
                               outputs=[cal_out])

            dict_status_banner = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    dict_original = gr.Textbox(label='🔤 النص الأصلي', interactive=False)
                    dict_correction = gr.Textbox(label='✏️ التصحيح المقترح')
                    dict_edit_btn = gr.Button('💾 تطبيق التعديل', variant='primary')
                with gr.Column(scale=1):
                    dict_meta = gr.Markdown()
                    dict_contexts = gr.Markdown()
            with gr.Row():
                prev_dict = gr.Button('⬅ السابق')
                reject_dict = gr.Button('🗑 رفض', variant='stop')
                approve_dict = gr.Button('✅ اعتماد', variant='primary')
                next_dict = gr.Button('التالي ➡')
            progress_dict = gr.Textbox(label='التقدم', interactive=False)

            _dict_outputs = [dict_status_banner, dict_original, dict_correction,
                             dict_meta, dict_contexts, progress_dict]

            def _load_wrap(p, l):
                res = load_dict_audit(priority=p, limit=int(l))
                if _dict_audit_state['rules']:
                    rule = _dict_audit_state['rules'][0]
                    status = calculate_rule_indicator(rule)
                    indicator = (f'<div style="padding:6px; background:{status["color"]}20; '
                                 f'border-radius:4px;">{status["indicator"]} — {status["summary"]}</div>')
                    return (indicator,) + res[:2] + res[2:6] + (res[6],)
                return ("", "", "", "", "", "0/0")

            load_audit_btn.click(_load_wrap, inputs=[priority_dropdown, limit_slider], outputs=_dict_outputs)
            priority_dropdown.change(_load_wrap, inputs=[priority_dropdown], outputs=_dict_outputs)
            dict_edit_btn.click(dict_audit_edit, inputs=[dict_correction], outputs=_dict_outputs)
            approve_dict.click(dict_audit_approve, outputs=_dict_outputs)
            reject_dict.click(dict_audit_reject, outputs=_dict_outputs)
            prev_dict.click(dict_audit_prev, outputs=_dict_outputs)
            next_dict.click(dict_audit_next, outputs=_dict_outputs)

    return app

In [ ]:
def launch():
    """تشغيل Gradio UI"""
    app = build_app()
    app.launch(
        share=CFG.gradio_share,
        server_port=0, # تم التعديل للسماح لـ Gradio باختيار منفذ متاح تلقائيًا
        quiet=False,
        theme=gr.themes.Soft(primary_hue='indigo') # نقل المعامل هنا حسب التحذير
    )
    return app


print()
print('✅✅✅ كل شيء جاهز — v5 Enhanced ✅✅✅')
print()
print('═' * 55)
print('  أوامر التشغيل:')
print('═' * 55)
print('  launch()                   ← Gradio UI كاملة (8 Tabs)')
print('  process_document()         ← معالجة بدون UI')
print('  MIGRATOR.migrate()         ← ترحيل من نسخ قديمة')
print('  auto_export("run_id")      ← تصدير CSV/XLSX/JSONL')
print('  generate_study_guide()     ← دليل الدراسة')
print('  finetune_trocr_lora()      ← LoRA Fine-tuning')
print('  active_learning_cycle()    ← دورة AL تلقائية')
print('  compute_metrics()          ← WER / CER')
print('  create_backup()            ← نسخة احتياطية')
print('═' * 55)
print()
print('  تلميح للذاكرة المنخفضة (16GB RAM):')
print('  CFG.trocr_batch_size = 1   ← معالجة كلمة بكلمة')
print('  CFG.dpi = 200              ← دقة أقل = ذاكرة أقل')
print('  CFG.pages_end = 2          ← صفحتان فقط للاختبار')

launch()

import shutil, os

drive_path = "/content/drive/MyDrive/HandwrittenOCR"
if os.path.exists(drive_path):
    total = 0
    for root, dirs, files in os.walk(drive_path):
        for f in files:
            fp = os.path.join(root, f)
            total += os.path.getsize(fp)
            size_mb = os.path.getsize(fp) / (1024*1024)
            if size_mb > 10:  # أظهر فقط الملفات أكبر من 10 ميجا
                print(f"  {size_mb:.1f} MB → {fp}")
    print(f"\nإجمالي الاستخدام: {total/(1024**3):.2f} جيجابايت")
else:
    print("المجلد غير موجود")